# **TEST DEAP PHIÊN BẢN 8**

In [1]:
import os
from google.colab import userdata

# 1. Lấy token và thông tin user
token = userdata.get('GITHUB_TOKEN')
username = "BonTapHoa" # Ví dụ: nguyenvanA
repo_name = "vietnam-satellite-simulation"       # Ví dụ: my-project
email = "tudinhbon2009@gmail.com"

# 2. Cấu hình đường dẫn Clone (đã bao gồm token để xác thực)
git_url = f"https://{token}@github.com/PTNK-ly-tin-2427/{repo_name}.git"

# 3. Clone repo về
!git clone {git_url}

# 4. Di chuyển vào thư mục repo
%cd "{repo_name}"

# 5. Cấu hình định danh (để GitHub biết ai commit)
!git config --global user.email {email}
!git config --global user.name {username}

# 6 checkout
!git checkout remotes/origin/moga-simulation-and-extract-plot

Cloning into 'vietnam-satellite-simulation'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 291 (delta 63), reused 164 (delta 43), pack-reused 100 (from 1)
Receiving objects: 100% (291/291), 5.33 MiB | 19.41 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/vietnam-satellite-simulation
Note: switching to 'remotes/origin/moga-simulation-and-extract-plot'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD i

In [2]:
!pip install deap skyfield tqdm matplotlib numpy pandas cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.0/367.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.7/235.7 kB 19.1 MB/s eta 0:00:00


In [3]:
# @title
# Import modules
from datetime import datetime, timedelta, UTC
from deap import base, tools, creator, algorithms
import numpy as np
import matplotlib.pyplot as plt
from skyfield.api import load, wgs84, EarthSatellite
import random
from tqdm import tqdm
import pandas as pd
import pickle
import os
import copy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [4]:
# @title
CUSTOM_LABELS = [
    "NSGA-NC",    # Test 0% [cite: 242, 312]
    "NSGA-C50",
    "NSGA-C60",
    "NSGA-C70",
    "NSGA-C80",
    "NSGA-C90",
    "NSGA-C95",
    "NSGA-C99",
    "NSGA-C100",
    "MOGA-WS"  # Test 2016 (MOGA trọng số) [cite: 312, 412]
]

In [5]:
# @title
# define the problem contraints
ARGUMENT = 6
POPULATION_SIZE = 50
CXPB = 0.8
MUTPB = 0.2
NGEN = 100
MIN_ACCEPTABLE_COVERAGE = 0.0

# Định nghĩa các biến gene và phạm vi
# [altitude, inclination, num_planes, sats_per_plane, phasing parameter]
BOUND_LOW = [500, 15, 1, 1, 1]
BOUND_UP = [1000, 25, 14, 24, 15]

# Define random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [6]:
# @title
# Thời gian mô phỏng 24 giờ, bước 1 phút
ts = load.timescale()
start_time = datetime(2025, 1, 1, 0, 0, 0, tzinfo=UTC) # Set start_time to Jan 1, 2025, 00:00:00 UTC
duration_hours = 12
time_step_minutes = 1

time_steps = []
current_time = start_time
while current_time <= start_time + timedelta(hours=duration_hours):
    time_steps.append(ts.utc(current_time))
    current_time += timedelta(minutes=time_step_minutes)

time_steps = np.array(time_steps)

In [9]:
# @title
# ID của Google Sheet
SHEET_ID = '1XmR-cF1an6RgFqF_wDUb6W0s6sEhuCtozxZVBEOowp4'
# Tên sheet (tab) cụ thể trong Google Sheet, nếu không có thì để trống
SHEET_NAME = '' # Để trống nếu chỉ có một sheet hoặc muốn lấy sheet đầu tiên

# Tạo URL để tải file CSV trực tiếp
# Sử dụng f-string để chèn SHEET_ID và SHEET_NAME
# Nếu SHEET_NAME rỗng, chỉ cần dùng export?format=csv&id=SHEET_ID
if SHEET_NAME:
    url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}'
else:
    url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv'

# Đọc dữ liệu từ URL vào DataFrame
try:
    df_ground_stations = pd.read_csv(url)
    print("✅ Đã tải dữ liệu thành công.")
    display(df_ground_stations.head())
except Exception as e:
    print(f"❌ Lỗi khi tải dữ liệu: {e}")
    print("Vui lòng kiểm tra lại ID Google Sheet và tên Sheet (nếu có).")


# Tạo mảng các trạm mặt đất từ DataFrame df_locations
ground_stations = np.array([wgs84.latlon(row['lat'], row['lon']) for index, row in df_ground_stations.iterrows()])

print(f"✅ Đã tạo {len(ground_stations)} trạm mặt đất từ dữ liệu.")

✅ Đã tải dữ liệu thành công.


,index,lon,lat
0,0,105.312949,9.0
1,1,106.576878,11.0
2,2,107.996440,11.0
3,3,107.850019,13.0
4,4,109.012039,13.0


✅ Đã tạo 15 trạm mặt đất từ dữ liệu.


## **4. Định nghĩa Hàm Fitness và Tạo Chòm sao**

In [7]:
# @title
import math

def mean_motion_from_altitude(altitude_km):
    """
    Tính mean motion (vòng/ngày) từ độ cao vệ tinh (km)
    theo định luật Kepler thứ ba.

    Parameters:
        altitude_km (float): Độ cao tính từ mặt đất (km)

    Returns:
        mean_motion_rev_per_day (float): Số vòng quay/ngày
    """
    mu = 398600.4418       # km^3/s^2
    R_E = 6378.137         # Bán kính Trái Đất (km)

    a = R_E + altitude_km  # Bán trục lớn (km)
    T = 2 * math.pi * math.sqrt(a**3 / mu)  # Chu kỳ (giây)
    mean_motion = 86400 / T                  # Vòng/ngày

    return mean_motion

In [8]:
# @title
def tle_checksum(line: str) -> str:
    checksum = 0
    # Chỉ duyệt qua 68 ký tự đầu tiên
    for char in line[:68]:
        if '0' <= char <= '9':
            checksum += int(char)
        elif char == '-':
            checksum += 1

    return str(checksum % 10)

In [9]:
# @title
from datetime import datetime, timezone

def tle_epoch_from_datetime(dt: datetime):
    """
    Chuyển đổi datetime UTC sang epoch TLE:
    - Trả về (epoch_year, epoch_day)
    - epoch_year: 2 chữ số cuối của năm
    - epoch_day: ngày trong năm + phần thập phân của ngày
    """
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    else:
        dt = dt.astimezone(timezone.utc)

    year = dt.year
    epoch_year = year % 100  # chỉ lấy 2 chữ số cuối

    # Tính ngày thứ DDD trong năm
    day_of_year = (dt - datetime(year, 1, 1, tzinfo=timezone.utc)).days + 1

    # Phần thập phân của ngày (giờ / 24)
    frac_day = (
        dt.hour / 24
        + dt.minute / 1440
        + dt.second / 86400
        + dt.microsecond / 8.64e10
    )

    epoch_day = day_of_year + frac_day
    return epoch_year, epoch_day

In [10]:
# @title
def generate_tle_celestrak(
    satnum, epoch_year, epoch_day,
    inclination_deg, raan_deg, eccentricity,
    arg_perigee_deg, mean_anomaly_deg,
    mean_motion_rev_per_day, rev_number=1
):
    # ---- LINE 1 ----
    line1 = (
        f"1 {satnum:05d}U 25001A   "
        f"{epoch_year:02d}{epoch_day:012.8f} "
        f" 0.00000000 00000-0 00000-0 0 0000"
    )
    line1 = line1[:68].ljust(68)
    line1 += tle_checksum(line1)

    # ---- LINE 2 ----
    line2 = (
        f"2 {satnum:05d} "
        f"{inclination_deg:8.4f} "
        f"{raan_deg:8.4f} "
        f"{int(eccentricity * 1e7):07d} "
        f"{arg_perigee_deg:8.4f} "
        f"{mean_anomaly_deg:8.4f} "
        f"{mean_motion_rev_per_day:11.8f}"
        f"{rev_number:5d}"
    )
    line2 = line2[:68].ljust(68)
    line2 += tle_checksum(line2)
    return line1, line2

In [11]:
# @title
def generate_constellation(individual):
  """
  Tạo mảng các vệ tinh EarthSatellite() từ cấu hình chòm sao.

  params = [altitude_km, inclination_deg, num_planes, sats_per_plane, phasing]
  """
  altitude_km, inclination_deg, num_planes, sats_per_plane, phasing = individual
  epoch_year, epoch_day = tle_epoch_from_datetime(start_time)
  mean_motion = mean_motion_from_altitude(altitude_km)
  eccentricity = 0.0
  arg_perigee_deg = 0.0
  satnum_base = 10000
  satellites = []
  for p in range(num_planes):
        RAAN = p * 360 / num_planes
        for s in range(sats_per_plane):
            mean_anomaly = (360 / sats_per_plane) * (
                s + phasing * p / num_planes
            ) % 360
            satnum = satnum_base + p * sats_per_plane + s + 1
            line1, line2 = generate_tle_celestrak(
                satnum=satnum,
                epoch_year=epoch_year,
                epoch_day=epoch_day,
                inclination_deg=inclination_deg,
                raan_deg=RAAN,
                eccentricity=eccentricity,
                arg_perigee_deg=arg_perigee_deg,
                mean_anomaly_deg=mean_anomaly,
                mean_motion_rev_per_day=mean_motion,
                rev_number=1
            )
            name = f"SAT_{satnum}"
            sat = EarthSatellite(line1, line2, name, ts)
            satellites.append(sat)

  return np.array(satellites)

In [12]:
# @title
from numpy import radians, degrees, arccos, dot
from numpy.linalg import norm

def constellation_coverage_vectorized(ground_stations, satellites, times, beamwidth_deg=360.0, min_elev_deg=25.0):
    """
    Phiên bản vector hóa: tính độ phủ sóng chòm sao vệ tinh theo thời gian.

    Parameters
    ----------
    ground_stations : np.ndarray
        Mảng các vị trí trạm mặt đất (đối tượng wgs84.latlon()).
    satellites : np.ndarray
        Mảng các đối tượng EarthSatellite.
    times : np.ndarray
        Mảng thời gian (ts.utc()).
    beamwidth_deg : float
        Góc mở chùm sóng vệ tinh (beamwidth).
    min_elev_deg : float
        Góc ngẩng tối thiểu.

    Returns
    -------
    coverage_ratio : float
        Tỷ lệ độ phủ sóng.
    coverage_matrix : np.ndarray
        Ma trận boolean [n_times, n_gs] — True nếu trạm được phủ tại thời điểm đó.
    """

    n_times = len(times)
    n_gs = len(ground_stations)
    n_sat = len(satellites)
    coverage_matrix = np.zeros((n_times, n_gs), dtype=bool)

    for t_idx, t in enumerate(times):
        # Lấy vị trí tất cả vệ tinh và GS tại thời điểm t
        sat_positions = np.array([sat.at(t).position.km for sat in satellites])     # (n_sat, 3)
        gs_positions  = np.array([gs.at(t).position.km for gs in ground_stations])  # (n_gs, 3)

        # ----------------------------
        # 1️⃣ Góc beam (beamwidth)
        # ----------------------------
        if beamwidth_deg >= 360.0:
            in_beam = True
        else:
            to_earth = -sat_positions
            to_earth_unit = to_earth / norm(to_earth, axis=1, keepdims=True)  # (n_sat, 3)

            # Tạo tensor (n_sat, n_gs, 3): vector từ vệ tinh -> trạm
            to_gs = gs_positions[None, :, :] - sat_positions[:, None, :]  # broadcasting
            norm_to_gs = norm(to_gs, axis=2, keepdims=True)
            to_gs_unit = to_gs / norm_to_gs

            # Góc giữa hướng beam và hướng đến trạm
            beam_cos = np.sum(to_earth_unit[:, None, :] * to_gs_unit, axis=2)
            beam_angle = degrees(np.arccos(np.clip(beam_cos, -1, 1)))
            in_beam = beam_angle <= beamwidth_deg # (n_sat, n_gs)

        # ----------------------------
        # 2️⃣ Góc ngẩng (elevation)
        # ----------------------------
        visible = True
        if min_elev_deg > 0.0:
            gs_norm = gs_positions / norm(gs_positions, axis=1, keepdims=True)  # (n_gs, 3)
            gs_to_sat = sat_positions[:, None, :] - gs_positions[None, :, :]     # (n_sat, n_gs, 3)

            dot_prod = np.sum(gs_to_sat * gs_norm[None, :, :], axis=2)
            elev_sin = dot_prod / norm(gs_to_sat, axis=2)
            elev_deg = degrees(np.arcsin(np.clip(elev_sin, -1, 1)))

            visible = (elev_deg >= min_elev_deg) & in_beam  # (n_sat, n_gs)

        # ----------------------------
        # 3️⃣ Một GS được phủ nếu có ít nhất 1 vệ tinh thỏa điều kiện
        # ----------------------------
        coverage_matrix[t_idx, :] = np.any(visible, axis=0)

    # ----------------------------
    # 4️⃣ Tính độ phủ sóng
    # ----------------------------
    coverage_ratio = np.sum(coverage_matrix) / (n_times * n_gs)
    return coverage_ratio

In [13]:
# @title
# Define a function to evaluate the total value of the selected items
def evaluate(individual):
    coverage = constellation_coverage_vectorized(ground_stations, generate_constellation(individual), time_steps)
    num_sats = individual[2] * individual[3]
    altitude = individual[0]
    violation = max(0, MIN_ACCEPTABLE_COVERAGE - coverage)
    individual.fitness.values = (coverage, altitude, num_sats)
    individual.fitness.cvalues = (violation,)
    return (coverage, altitude, num_sats)  # Maximize coverage and minimize num_sats, altitude

In [14]:
# @title
def constrained_dominates(fit1, fit2):
    """Trả về True nếu ind1 thống trị ind2 (có xét ràng buộc), ngược lại False."""

    # Lấy tổng vi phạm của mỗi cá thể
    v1, = fit1.cvalues
    v2, = fit2.cvalues

    # Quy tắc 1: ind1 khả thi, ind2 không khả thi -> ind1 thắng
    if v1 == 0 and v2 > 0:
        return True
    # Quy tắc 2: ind1 không khả thi, ind2 khả thi -> ind1 thua
    elif v1 > 0 and v2 == 0:
        return False
    # Quy tắc 3: Cả hai đều không khả thi
    elif v1 > 0 and v2 > 0:
        # Cá thể nào vi phạm ít hơn thì thắng
        return bool(v1 < v2)
    # Quy tắc 4: Cả hai đều khả thi (v1 == 0 and v2 == 0)
    else:
        # Dùng luật thống trị Pareto chuẩn (DEAP đã có sẵn)
        return fit1.dominates(fit2)

In [15]:
# @title
from collections import defaultdict, namedtuple
from itertools import chain
from operator import attrgetter, itemgetter

def fastConstrainedNondominatedSort(individuals, k, first_front_only=False):
    if k == 0:
          return []
    map_fit_ind = defaultdict(list)
    for ind in individuals:
        map_fit_ind[ind.fitness].append(ind)
    fits = list(map_fit_ind.keys())

    current_front = []
    next_front = []
    dominating_fits = defaultdict(int)
    dominated_fits = defaultdict(list)

    # Rank first Pareto front
    for i, fit_i in enumerate(fits):
        for fit_j in fits[i + 1:]:
            if constrained_dominates(fit_i, fit_j):
                dominating_fits[fit_j] += 1
                dominated_fits[fit_i].append(fit_j)
            elif constrained_dominates(fit_j, fit_i):
                dominating_fits[fit_i] += 1
                dominated_fits[fit_j].append(fit_i)
        if dominating_fits[fit_i] == 0:
            current_front.append(fit_i)

    fronts = [[]]
    for fit in current_front:
        fronts[-1].extend(map_fit_ind[fit])
    pareto_sorted = len(fronts[-1])

    # Rank the next front until all individuals are sorted or
    # the given number of individual are sorted.
    if not first_front_only:
        N = min(len(individuals), k)
        while pareto_sorted < N:
            fronts.append([])
            for fit_p in current_front:
                for fit_d in dominated_fits[fit_p]:
                    dominating_fits[fit_d] -= 1
                    if dominating_fits[fit_d] == 0:
                        next_front.append(fit_d)
                        pareto_sorted += len(map_fit_ind[fit_d])
                        fronts[-1].extend(map_fit_ind[fit_d])
            current_front = next_front
            next_front = []

    return fronts

def selConstrainedNSGA2(individuals, k):
    pareto_fronts = fastConstrainedNondominatedSort(individuals, k)
    for front in pareto_fronts:
        tools.emo.assignCrowdingDist(front)

    chosen = list(chain(*pareto_fronts[:-1]))
    k = k - len(chosen)
    if k > 0:
        sorted_front = sorted(pareto_fronts[-1], key=attrgetter("fitness.crowding_dist"), reverse=True)
        chosen.extend(sorted_front[:k])

    return chosen

In [16]:
# @title
from re import I
import random

# Create the fitness class
creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)

# Create the individual class
creator.create("Individual", list, fitness=creator.FitnessMulti)

# Register the individualCreator operator
def init_individual():
    altitude = random.randint(BOUND_LOW[0], BOUND_UP[0])
    inclination = random.randint(BOUND_LOW[1], BOUND_UP[1])
    num_planes = random.randint(BOUND_LOW[2], BOUND_UP[2])
    sats_per_plane = random.randint(BOUND_LOW[3], BOUND_UP[3])
    phasing = 1
    if BOUND_LOW[4] < num_planes-1:
      phasing = random.randint(BOUND_LOW[4], num_planes-1)
    return creator.Individual(
        [altitude, inclination, num_planes, sats_per_plane, phasing]
    )

toolbox = base.Toolbox()
toolbox.register(
    "individualCreator",
    init_individual
)

# Register the populationCreator operator
toolbox.register("populationCreator", tools.initRepeat, list, toolbox.individualCreator)

toolbox.register("evaluate", evaluate)

# Create the genetic operators
toolbox.register("select", selConstrainedNSGA2)
def custom_mate(ind1, ind2):
  size = min(len(ind1), len(ind2))
  cxpoint1 = random.randint(1, size)
  cxpoint2 = random.randint(1, size - 1)
  if cxpoint2 >= cxpoint1:
      cxpoint2 += 1
  else:  # Swap the two cx points
      cxpoint1, cxpoint2 = cxpoint2, cxpoint1

  ind1[cxpoint1:cxpoint2], ind2[cxpoint1:cxpoint2] \
      = ind2[cxpoint1:cxpoint2], ind1[cxpoint1:cxpoint2]
  ind1[4] = min(ind1[2] - 1, ind1[4])
  ind2[4] = min(ind2[2] - 1, ind2[4])

  return ind1, ind2

toolbox.register("mate", custom_mate)  # crossover

def custom_mutation(individual, low, up, indpb):
  for i in range(len(individual)):
    if random.random() < indpb:
      if i == 4:
        individual[i] = 1
        if low[i] < individual[2] - 1:
          individual[i] = random.randint(low[i], individual[2] - 1)
      else:
        individual[i] = random.randint(low[i], up[i])
  individual[4] = min(individual[2] - 1, individual[4])
  return individual,

toolbox.register(
    "mutate",
    custom_mutation,
    low=BOUND_LOW,
    up=BOUND_UP,
    indpb=0.5,
)

## **6. Lưu trữ và Tải Checkpoint (Saving and Loading Checkpoints) 💾**

In [17]:
# @title
# Đảm bảo thư mục checkpoint tồn tại trong Google Drive
from google.colab import drive
os.makedirs('/content/drive', exist_ok=True)
drive.mount('/content/drive')

# Đường dẫn tới thư mục checkpoint trên Google Drive
# Thay thế 'GA_Checkpoints' bằng tên thư mục bạn muốn lưu trong MyDrive
DRIVE_CHECKPOINT_FOLDER = '/content/drive/MyDrive/PTNK/NCKH/Sprint 11'
CHECKPOINT_FILE = [os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_0.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_50.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_60.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_70.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_80.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_90.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_95.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_99.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_100.pkl"),
                   os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_2016.pkl"),]

OUTPUT_FOLDER = '/content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15'

# Tạo thư mục nếu nó chưa tồn tại
os.makedirs(DRIVE_CHECKPOINT_FOLDER, exist_ok=True)

Mounted at /content/drive


In [18]:
CHECKPOINT_FILES =[0, 50, 60, 70, 80, 90, 95, 99, 100, 2016]

## **7. Chạy Thuật toán (Running the Algorithm) ⚙️**

In [ ]:
# # define the algorithm
# def run_algorithm(population, toolbox, mu, lambda_, cxpb, mutpb, ngen,
#                    logbook=None, stats=None, halloffame=None, halloffame_history=None, pop_history=None, verbose=__debug__):
#     if os.path.exists(CHECKPOINT_FILE):
#         with open(CHECKPOINT_FILE, "rb") as cp_file:
#             cp = pickle.load(cp_file)
#         population[:] = cp["population"]
#         gen_start = cp["generation"]
#         halloffame = cp["halloffame"]
#         logbook[:] = cp["logbook"]
#         halloffame_history = cp['halloffame_history']
#         pop_history = cp['pop_history']
#         print(f"▶️ Tiếp tục từ thế hệ {gen_start}")
#     else:
#         gen_start = 0
#         logbook.header = ['gen', 'nevals', 'new_hof'] + (stats.fields if stats else [])
#         halloffame_history = []

#     # Đánh giá quần thể khởi tạo
#     if gen_start == 0: # Chỉ đánh giá quần thể khởi tạo nếu bắt đầu từ thế hệ 0
#         # Evaluate the individuals with an invalid fitness
#         invalid_ind = [ind for ind in population if not ind.fitness.valid]
#         for ind in tqdm(invalid_ind, desc="Đánh giá quần thể khởi tạo"):
#             ind.fitness.values = toolbox.evaluate(ind)

#         # Sau khi đánh giá, cần chạy select một lần để gán dominance_class và distance cho front 0
#         population = toolbox.select(population, len(population))

#         if halloffame is not None:
#             halloffame.update(population)

#         # 💾 Lưu toàn bộ halloffame ở thời điểm hiện tại
#         hof_snapshot = {
#             "gen": 0,
#             "halloffame": [list(ind) for ind in halloffame],
#             "fitnesses": [tuple(ind.fitness.values) for ind in halloffame]
#         }
#         # copy để tránh bị thay đổi bởi DEAP
#         halloffame_history.append(copy.deepcopy(hof_snapshot))

#         # Lưu POPULATION HISTORY
#         pop_snapshot = {
#             "gen": 0,
#             "population": [list(ind) for ind in population],
#             "fitnesses": [tuple(ind.fitness.values) for ind in population]
#         }
#         pop_history.append(copy.deepcopy(pop_snapshot))

#         # Ghi nhận thống kê cho thế hệ 0
#         record = stats.compile(population) if stats is not None else {}
#         logbook.record(gen=0, nevals=len(invalid_ind), new_hof=len(halloffame), **record)
#         if verbose:
#             print(logbook.stream)

#         # Lưu checkpoint sau khi đánh giá quần thể khởi tạo
#         cp = dict(population=population, generation=0, halloffame=halloffame, logbook=logbook, halloffame_history=halloffame_history, pop_history=pop_history)
#         with open(CHECKPOINT_FILE, "wb") as cp_file:
#                 pickle.dump(cp, cp_file)
#         print(f"✅ Đã lưu checkpoint thế hệ 0 vào {CHECKPOINT_FILE}")

#     # Begin the generational process
#     for gen in range(gen_start+1, ngen+1):
#         # # Select the next generation individuals
#         offspring = algorithms.varOr(population, toolbox, lambda_, cxpb, mutpb)

#         # Đánh giá cá thể chưa có fitness
#         invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
#         for ind in tqdm(invalid_ind, desc=f"Đánh giá thế hệ {gen}"):
#             ind.fitness.values = toolbox.evaluate(ind)

#         # Select the next generation population
#         population[:] = toolbox.select(population + offspring, mu)

#         # Xác định halloffame mới cập nhật
#         if halloffame is not None:
#             prev_hof_set = {tuple(ind) for ind in halloffame}
#             halloffame.update(offspring)
#             current_hof_set = {tuple(ind) for ind in halloffame}
#             new_hof_individuals = len(current_hof_set.difference(prev_hof_set))

#         # 💾 Update halloffame history
#         halloffame_snapshot = {
#             "gen": gen,
#             "halloffame": [list(ind) for ind in halloffame],
#             "fitnesses": [tuple(ind.fitness.values) for ind in halloffame]
#         }
#         halloffame_history.append(copy.deepcopy(halloffame_snapshot))

#         # Lưu POPULATION HISTORY
#         pop_snapshot = {
#             "gen": gen,
#             "population": [list(ind) for ind in population],
#             "fitnesses": [tuple(ind.fitness.values) for ind in population]
#         }
#         pop_history.append(copy.deepcopy(pop_snapshot))

#         # Append the current generation statistics to the logbook
#         record = stats.compile(population) if stats is not None else {}
#         logbook.record(gen=gen, nevals=len(invalid_ind), new_hof=new_hof_individuals, **record)
#         if verbose:
#             print(logbook.stream)

#         # Lưu checkpoint sau mỗi thế hệ
#         cp = dict(population=population, generation=gen, halloffame=halloffame, logbook=logbook, halloffame_history=halloffame_history, pop_history=pop_history)
#         with open(CHECKPOINT_FILE, "wb") as cp_file:
#             pickle.dump(cp, cp_file)
#         print(f"✅ Đã lưu checkpoint thế hệ {gen} vào {CHECKPOINT_FILE}")

In [ ]:
# pop = toolbox.populationCreator(n=POPULATION_SIZE)
# logbook = tools.Logbook()
# halloffame = tools.ParetoFront()
# halloffame_history = []
# pop_history = []

# if os.path.exists(CHECKPOINT_FILE):
#   with open(CHECKPOINT_FILE, "rb") as cp_file:
#       cp = pickle.load(cp_file)
#   pop = cp["population"]
#   halloffame = cp["halloffame"]
#   logbook = cp["logbook"]
#   halloffame_history = cp['halloffame_history']
#   pop_history = cp['pop_history']

# # Create the statistics objects for độ phủ
# stats_coverage = tools.Statistics(lambda ind: ind.fitness.values[0])
# stats_coverage.register("avg", np.mean)
# stats_coverage.register("std", np.std)
# stats_coverage.register("min", np.min)
# stats_coverage.register("max", np.max)

# stats_costs = tools.Statistics(lambda ind: ind.fitness.values[1])
# stats_costs.register("avg", np.mean)
# stats_costs.register("std", np.std)
# stats_costs.register("min", np.min)
# stats_costs.register("max", np.max)

# stats_altitude = tools.Statistics(lambda ind: ind.fitness.values[2])
# stats_altitude.register("avg", np.mean)
# stats_altitude.register("std", np.std)
# stats_altitude.register("min", np.min)
# stats_altitude.register("max", np.max)

# # Thống kê chung
# stats = tools.MultiStatistics(obj1=stats_coverage, obj2=stats_costs, obj3=stats_altitude)
# stats.register("avg", np.mean)
# stats.register("std", np.std)
# stats.register("min", np.min)
# stats.register("max", np.max)

# run_algorithm(pop, toolbox,
#               mu=POPULATION_SIZE,
#               lambda_=POPULATION_SIZE,
#               cxpb=CXPB,
#               mutpb=MUTPB,
#               ngen=NGEN,
#               logbook=logbook,
#               stats=stats,
#               halloffame=halloffame,
#               halloffame_history=halloffame_history,
#               pop_history=pop_history,
#               verbose=True)

##LOAD

In [19]:
# @title
# Dictionary to store data loaded from different checkpoints
loaded_checkpoints_data = {}

import re

for i, checkpoint_file in enumerate(CHECKPOINT_FILE):
  if os.path.exists(checkpoint_file):
    try:
      # Extract the integer from the filename
      match = re.search(r'_(\d+)\.pkl$', os.path.basename(checkpoint_file))
      if match:
          checkpoint_key = int(match.group(1))
      else:
          # Handle cases where the filename doesn't match the expected pattern, e.g., use the full name or an index
          # match = re.search(r'_(\w+)\.pkl$', os.path.basename(checkpoint_file))
          checkpoint_key = "roullete"
          print(f"⚠️ Could not extract integer from filename: {os.path.basename(checkpoint_file)}. Using full filename as key.")


      with open(checkpoint_file, "rb") as cp_file:
          cp = pickle.load(cp_file)
          # Store the loaded data, associating it with the extracted key
          loaded_checkpoints_data[checkpoint_key] = {
              "population": cp["population"],
              "halloffame": cp["halloffame"],
              "logbook": cp["logbook"],
              "halloffame_history": cp['halloffame_history'],
              "pop_history": cp['pop_history']
          }
      print(f"✅ Đã tải dữ liệu thành công từ checkpoint: {checkpoint_file} với key: {checkpoint_key}")
    except Exception as e:
      print(f"❌ Lỗi khi tải dữ liệu từ checkpoint {checkpoint_file}: {e}")
  else:
    print(f"⚠️ Tệp checkpoint không tồn tại: {checkpoint_file}")

# Now loaded_checkpoints_data contains the data from each file, indexed by the integer from the filename.
# You can access the data for a specific file like: loaded_checkpoints_data[0] or loaded_checkpoints_data[80] etc.

✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_0.pkl với key: 0
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_50.pkl với key: 50
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_60.pkl với key: 60
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_70.pkl với key: 70
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_80.pkl với key: 80
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_90.pkl với key: 90
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_final_constraint_95.pkl với key: 95
✅ Đã tải dữ liệu thành công từ checkpoint: /content/drive/MyDrive/PTNK/

In [20]:
# @title
with open(os.path.join(DRIVE_CHECKPOINT_FOLDER, "checkpoint_final_constraint_2016.pkl"), "rb") as cp_file:
  cp = pickle.load(cp_file)
  # Store the loaded data, associating it with the extracted key
  loaded_checkpoints_data[2016] = {
      "population": cp["population"],
      "halloffame": cp["halloffame"],
      "logbook": cp["logbook"],
      "halloffame_history": cp['halloffame_history'],
      "pop_history": cp['pop_history']
  }

In [21]:
# @title
import pandas as pd

# Create a list to hold the data for the DataFrame
pop_history_data = []

# Iterate through the pop_history (list of dictionaries, one per generation)
for gen_snapshot in loaded_checkpoints_data[2016]['pop_history']:
    gen = gen_snapshot["gen"]
    population_list = gen_snapshot["halloffame"]
    fitnesses_main_list = gen_snapshot["fitness main"]
    # Access fitness coverage values from the correct key
    fitnesses_coverage_list = gen_snapshot["fitness coverage"]


    # Iterate through each individual in the population for this generation
    for i in range(len(population_list)):
        individual = population_list[i]
        fitness_main_values = fitnesses_main_list[i]
        fitness_coverage_values = fitnesses_coverage_list[i]

        # Calculate Fitness (Altitude) and Fitness (Costs) based on individual parameters and weights
        altitude = individual[0]
        num_sats = individual[2] * individual[3]

        # Note: These calculations might not exactly match the values stored in ind.fitness.cvalues
        # if the evaluate function uses a different formula or scaling before storing.
        # However, this is the best we can do without modifying the run_algorithm to store more values.
        fitness_altitude = altitude
        fitness_costs = num_sats


        # Append the flattened data as a dictionary to the list
        pop_history_data.append({
            "Generation": gen,
            "Individual_Index": i + 1,
            "Altitude (km)": individual[0],
            "Inclination (deg)": individual[1],
            "Số mặt phẳng quỹ đạo": int(individual[2]),
            "Số vệ tinh mỗi mặt phẳng": int(individual[3]),
            "Phasing": individual[4],
            # "Fitness (Main)": fitness_main_values,
            "Fitness (Coverage)": fitness_coverage_values,
            "Fitness (Altitude)": fitness_altitude,
            "Fitness (Costs)": fitness_costs
        })

# Create the pandas DataFrame from the flattened data
loaded_checkpoints_data[2016]['pop_history'] = pd.DataFrame(pop_history_data)

In [22]:
# @title
import pandas as pd
from collections import defaultdict

def convert_standardized_df_to_deap_history(standardized_df):
    """
    Converts a standardized DataFrame (like the one created previously for key 2016)
    back to the original DEAP history format (list of dictionaries per generation).

    Parameters
    ----------
    standardized_df : pandas.DataFrame
        The DataFrame containing history data with columns like
        'Generation', 'Altitude (km)', ..., 'Fitness (Coverage)', ...

    Returns
    -------
    list
        A list of dictionaries, where each dictionary represents a generation
        and contains keys 'gen', 'population' (list of gene lists), and
        'fitnesses' (list of fitness tuples).
    """
    if standardized_df.empty:
        return []

    deap_history = []
    # Group the DataFrame by Generation
    grouped_by_gen = standardized_df.groupby('Generation')

    for gen, gen_df in grouped_by_gen:
        population = []
        fitnesses = []
        for index, row in gen_df.iterrows():
            # Reconstruct the individual's gene values
            individual_genes = [
                row['Altitude (km)'],
                row['Inclination (deg)'],
                row['Số mặt phẳng quỹ đạo'],
                row['Số vệ tinh mỗi mặt phẳng'],
                row['Phasing']
            ]
            population.append(individual_genes)

            # Reconstruct the individual's fitness tuple
            fitness_values = (
                row['Fitness (Coverage)'],
                row['Fitness (Altitude)'],
                row['Fitness (Costs)']
            )
            fitnesses.append(fitness_values)

        # Append the data for this generation
        deap_history.append({
            'gen': gen,
            'population': population, # Store list of gene lists
            'fitnesses': fitnesses # Store list of fitness tuples
            # Note: 'new_hof' and other logbook stats are not part of this structure,
            # they are typically derived from the logbook itself or calculated separately.
            # We are aiming for the structure needed by analyze functions like
            # count_100_percent_coverage_individuals.
        })

    return deap_history

# Assuming loaded_checkpoints_data is already populated and contains key 2016
checkpoint_key_to_standardize_back = 2016

if checkpoint_key_to_standardize_back in loaded_checkpoints_data:
    data_to_convert = loaded_checkpoints_data[checkpoint_key_to_standardize_back]

    # Convert pop_history if it's currently a DataFrame
    if isinstance(data_to_convert.get('pop_history'), pd.DataFrame):
        print(f"Converting pop_history DataFrame back to DEAP history format for key {checkpoint_key_to_standardize_back}")
        loaded_checkpoints_data[checkpoint_key_to_standardize_back]['pop_history'] = convert_standardized_df_to_deap_history(data_to_convert['pop_history'])
        print(f"✅ Converted pop_history for key {checkpoint_key_to_standardize_back}.")
    else:
        print(f"⚠️ pop_history for key {checkpoint_key_to_standardize_back} is not a DataFrame. No conversion needed.")


    # # Convert halloffame_history if it's currently a DataFrame
    # if isinstance(data_to_convert.get('halloffame_history'), pd.DataFrame):
    #     print(f"Converting halloffame_history DataFrame back to DEAP history format for key {checkpoint_key_to_standardize_back}")
    #     loaded_checkpoints_data[checkpoint_key_to_standardize_back]['halloffame_history'] = convert_standardized_df_to_deap_history(data_to_convert['halloffame_history'])
    #     print(f"✅ Converted halloffame_history for key {checkpoint_key_to_standardize_back}.")
    # else:
    #      print(f"⚠️ halloffame_history for key {checkpoint_key_to_standardize_back} is not a DataFrame. No conversion needed.")

else:
    print(f"⚠️ Checkpoint key '{checkpoint_key_to_standardize_back}' not found in loaded_checkpoints_data.")

# You can now verify the format by inspecting loaded_checkpoints_data[2016]['pop_history']
# and loaded_checkpoints_data[2016]['halloffame_history']
print("\nVerifying format after conversion:")
if 2016 in loaded_checkpoints_data:
    if 'pop_history' in loaded_checkpoints_data[2016]:
        print("\nSample of pop_history:")
        print(loaded_checkpoints_data[2016]['pop_history'][:2]) # Print first 2 generations
    # if 'halloffame_history' in loaded_checkpoints_data[2016]:
    #     print("\nSample of halloffame_history:")
    #     print(loaded_checkpoints_data[2016]['halloffame_history'][:2]) # Print first 2 generations

Converting pop_history DataFrame back to DEAP history format for key 2016
✅ Converted pop_history for key 2016.

Verifying format after conversion:

Sample of pop_history:
[{'gen': 0, 'population': [[np.float64(552.0), np.float64(25.0), np.float64(12.0), np.float64(18.0), np.float64(2.0)], [np.float64(943.0), np.float64(16.0), np.float64(7.0), np.float64(9.0), np.float64(4.0)], [np.float64(513.0), np.float64(23.0), np.float64(4.0), np.float64(23.0), np.float64(3.0)], [np.float64(552.0), np.float64(25.0), np.float64(12.0), np.float64(18.0), np.float64(2.0)], [np.float64(635.0), np.float64(15.0), np.float64(12.0), np.float64(15.0), np.float64(9.0)], [np.float64(691.0), np.float64(22.0), np.float64(9.0), np.float64(15.0), np.float64(2.0)], [np.float64(821.0), np.float64(17.0), np.float64(13.0), np.float64(22.0), np.float64(7.0)], [np.float64(691.0), np.float64(22.0), np.float64(9.0), np.float64(15.0), np.float64(2.0)], [np.float64(739.0), np.float64(23.0), np.float64(5.0), np.float64(18.0

In [23]:
# @title
import copy

def reconstruct_halloffame_history_from_pop_history(pop_history, constrained_dominates_func, sort_func):
    """
    Reconstructs the halloffame_history from a pop_history that is in the
    original DEAP list-of-dictionaries format.

    Parameters
    ----------
    pop_history : list
        A list of dictionaries, where each dictionary represents a generation's
        data, including 'gen', 'population' (list of gene lists), and
        'fitnesses' (list of fitness tuples).
    constrained_dominates_func : function
        The function used to determine constrained dominance (e.g., constrained_dominates).
    sort_func : function
        The non-dominated sorting function (e.g., fastConstrainedNondominatedSort).

    Returns
    -------
    list
        A list of dictionaries representing the halloffame_history,
        with keys 'gen', 'halloffame' (list of gene lists), and
        'fitnesses' (list of fitness tuples).
    """
    reconstructed_hof_history = []
    temp_hof = tools.ParetoFront()

    if not pop_history:
        print("⚠️ pop_history is empty. Cannot reconstruct halloffame_history.")
        return reconstructed_hof_history

    # Ensure DEAP creator is set up
    # if not hasattr(creator, "FitnessMulti"):
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
    # if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMulti)

    for gen_data in tqdm(pop_history, desc="Reconstructing Hall of Fame History"):
        generation = gen_data['gen']
        population_genes = gen_data['population']
        fitnesses = gen_data['fitnesses']

        # Create DEAP individuals for the current generation's population
        current_population = []
        for i in range(len(population_genes)):
            ind = creator.Individual(population_genes[i])
            ind.fitness.values = fitnesses[i]
            # Assuming cvalues are not stored, or can be derived if needed.
            # For NSGA-II's Pareto front, dominance is key, which uses fitness.values.
            # If constrained dominance is crucial for the HOF definition, cvalues might need to be re-evaluated or handled.
            # Based on the user's desired output format, only fitnesses are explicitly included.
            # Let's add a placeholder cvalue if it's required by the sort_func.
            ind.fitness.cvalues = (0,) # Placeholder - adjust if actual constraint violation matters
            current_population.append(ind)
        # Apply non-dominated sorting to find the Pareto front (Hall of Fame) for this generation
        # We want only the first front
        temp_hof.update(current_population)

        # Extract gene values and fitnesses from the HOF individuals
        hof_gene_values = [list(ind) for ind in temp_hof]
        hof_fitness_values = [ind.fitness.values for ind in temp_hof]

        # Append the HOF data for this generation
        reconstructed_hof_history.append({
            'gen': generation,
            'halloffame': hof_gene_values,
            'fitnesses': hof_fitness_values
        })

    return reconstructed_hof_history

# Example usage:
# Assuming loaded_checkpoints_data[2016]['pop_history'] is in the correct DEAP history format
# and constrained_dominates and fastConstrainedNondominatedSort are defined.

roullete_key = 2016
if roullete_key in loaded_checkpoints_data and isinstance(loaded_checkpoints_data[roullete_key].get('pop_history'), list):
    print(f"Reconstructing halloffame_history for checkpoint key: {roullete_key}")
    reconstructed_hof_history = reconstruct_halloffame_history_from_pop_history(
        loaded_checkpoints_data[roullete_key]['pop_history'],
        constrained_dominates, # Pass the constrained_dominates function
        fastConstrainedNondominatedSort # Pass the sorting function
    )

    # Replace the existing halloffame_history with the reconstructed one
    loaded_checkpoints_data[roullete_key]['halloffame_history'] = reconstructed_hof_history

    print(f"✅ Reconstructed and replaced halloffame_history for checkpoint {roullete_key}.")

    # Display a sample of the reconstructed halloffame_history to verify
    print("\nSample of reconstructed halloffame_history:")
    # v # Display first 2 generations

else:
    print(f"⚠️ pop_history for checkpoint key {roullete_key} not found or is not in the correct format. Skipping reconstruction.")

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMulti' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


Reconstructing halloffame_history for checkpoint key: 2016


Reconstructing Hall of Fame History: 100%|██████████| 101/101 [00:00<00:00, 499.78it/s]

✅ Reconstructed and replaced halloffame_history for checkpoint 2016.

Sample of reconstructed halloffame_history:


In [24]:
# @title
import pandas as pd
import numpy as np

def convert_pop_history_to_dataframe(pop_history_list):
    """
    Converts a DEAP pop_history (list of generation snapshots) into a single
    pandas DataFrame, flattening the individual and fitness data.

    Parameters
    ----------
    pop_history_list : list
        A list of dictionaries, where each dictionary represents a generation
        and contains keys 'gen', 'population', and 'fitnesses'.

    Returns
    -------
    pandas.DataFrame
        A DataFrame where each row represents an individual from a specific
        generation, with columns for gene values and fitness values.
    """
    flattened_data = []
    for gen_snapshot in pop_history_list:
        gen = gen_snapshot["gen"]
        population = gen_snapshot["population"]
        fitnesses = gen_snapshot["fitnesses"]

        for i in range(len(population)):
            individual_genes = population[i]
            individual_fitness = fitnesses[i]

            # Ensure individual_genes has 5 elements: altitude, inclination, num_planes, sats_per_plane, phasing
            if len(individual_genes) >= 5:
                altitude, inclination, num_planes, sats_per_plane, phasing = individual_genes[0], individual_genes[1], individual_genes[2], individual_genes[3], individual_genes[4]
            else:
                # Handle cases where individual_genes might be shorter if the structure changed or is incomplete
                altitude, inclination, num_planes, sats_per_plane, phasing = [np.nan] * 5

            # Ensure individual_fitness has 3 elements: coverage, altitude_fitness, num_sats_fitness
            if len(individual_fitness) >= 3:
                coverage, altitude_fitness, num_sats_fitness = individual_fitness[0], individual_fitness[1], individual_fitness[2]
            else:
                coverage, altitude_fitness, num_sats_fitness = [np.nan] * 3

            flattened_data.append({
                "generation": gen,
                "index": i + 1,
                "coverage": coverage,
                "altitude": altitude,
                "num_sats": int(num_sats_fitness) if not pd.isna(num_sats_fitness) else np.nan,
                "inclination": inclination,
                "num_planes": int(num_planes) if not pd.isna(num_planes) else np.nan,
                "num_sats_per_plane": int(sats_per_plane) if not pd.isna(sats_per_plane) else np.nan,
                "phasing": phasing,
            })
    return pd.DataFrame(flattened_data)

# List of checkpoint keys to process
checkpoint_keys_to_process = [0, 50, 60, 70, 80, 90, 95, 99, 100, 2016]

# Dictionary to store the resulting DataFrames
pop_history_dataframes = {}

if 'loaded_checkpoints_data' in locals():
    for key in checkpoint_keys_to_process:
        if key in loaded_checkpoints_data and 'pop_history' in loaded_checkpoints_data[key]:
            print(f"Converting pop_history for checkpoint key: {key}...")
            # Ensure pop_history is in the correct list format for conversion
            pop_history_data = loaded_checkpoints_data[key]['pop_history']

            # If pop_history is already a DataFrame, convert it back to the list format first
            # This handles cases where it might have been converted to DF in a previous step
            if isinstance(pop_history_data, pd.DataFrame):
                 # Assuming the DataFrame has columns 'Generation', 'Individual_Index', etc.
                 # as created by the previous standardization step.
                 # Reconstruct to list of dict format for the conversion function.
                 # This part is complex and assumes the structure is recoverable, but for now,
                 # we'll assume it's already in the expected list of dicts from `loaded_checkpoints_data`.
                 # If it needs to be reconstructed, `convert_standardized_df_to_deap_history` would be used.
                 # However, the user's prompt implies the original list of dicts.
                 print(f"Warning: pop_history for key {key} is already a DataFrame. Assuming the previous conversion (cell 2Vr5yNubmot0) was a temporary step. If not, re-run cell 2Vr5yNubmot0 to convert it back correctly.")
                 # Re-calling `convert_standardized_df_to_deap_history` if needed to ensure list format:
                 # If it was a dataframe, it means `loaded_checkpoints_data[key]['pop_history']` was replaced by a dataframe.
                 # The original `convert_standardized_df_to_deap_history` function was used to convert a pandas DataFrame back to a DEAP history list.
                 # If `loaded_checkpoints_data[key]['pop_history']` is still the DataFrame:
                 pop_history_data = convert_standardized_df_to_deap_history(pop_history_data)

            df = convert_pop_history_to_dataframe(pop_history_data)
            pop_history_dataframes[key] = df
            print(f"✅ Successfully converted pop_history for checkpoint key {key}.")
        else:
            print(f"⚠️ pop_history not found for checkpoint key: {key}. Skipping.")
    print("--- All pop_history DataFrames converted and stored in `pop_history_dataframes` ---")
else:
    print("Error: `loaded_checkpoints_data` not found. Please ensure checkpoints are loaded.")

# Display the head of the DataFrame for key 0 as an example
if 0 in pop_history_dataframes:
    print("\nExample DataFrame for checkpoint key 0:")
    display(pop_history_dataframes[0].head())


Converting pop_history for checkpoint key: 0...
✅ Successfully converted pop_history for checkpoint key 0.
Converting pop_history for checkpoint key: 50...
✅ Successfully converted pop_history for checkpoint key 50.
Converting pop_history for checkpoint key: 60...
✅ Successfully converted pop_history for checkpoint key 60.
Converting pop_history for checkpoint key: 70...
✅ Successfully converted pop_history for checkpoint key 70.
Converting pop_history for checkpoint key: 80...
✅ Successfully converted pop_history for checkpoint key 80.
Converting pop_history for checkpoint key: 90...
✅ Successfully converted pop_history for checkpoint key 90.
Converting pop_history for checkpoint key: 95...
✅ Successfully converted pop_history for checkpoint key 95.
Converting pop_history for checkpoint key: 99...
✅ Successfully converted pop_history for checkpoint key 99.
Converting pop_history for checkpoint key: 100...
✅ Successfully converted pop_history for checkpoint key 100.
Converting pop_hist

,generation,index,coverage,altitude,num_sats,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,0.998890,552,216,25,12,18,2
1,0,2,0.009986,802,1,21,1,1,1
2,0,3,0.707166,547,68,18,4,17,3
3,0,4,0.833934,513,92,23,4,23,3
4,0,5,0.947850,859,56,23,7,8,4


## **8. Phân tích quá trình tiến hoá**

Phần này trình bày cách ghi nhận, theo dõi và trực quan hoá tiến trình tiến hoá của quần thể trong quá trình tối ưu đa mục tiêu bằng thư viện DEAP. Hai nguồn dữ liệu chính được sử dụng là Logbook – ghi lại thống kê ở từng thế hệ, và Hall of Fame History – lưu trữ lịch sử các cá thể ưu tú nhất trong suốt quá trình tiến hoá.

### **8.1 Nhật ký Tiến hóa của Quần thể (Logbook)**
hật ký tiến hoá (Logbook) được tạo tự động trong quá trình chạy thuật toán. Mỗi thế hệ (generation) được ghi lại các chỉ số quan trọng như:

*   Số lượng cá thể được đánh giá (nevals)
*   Giá trị trung bình, nhỏ nhất và lớn nhất của từng mục tiêu (obj1, obj2, obj3, …)
*   Độ lệch chuẩn của từng mục tiêu (std)

Các dữ liệu này được chuyển thành DataFrame để dễ dàng phân tích và trực quan hoá:

In [51]:
# @title
import pandas as pd

def logbook_to_dataframe(logbook):
    """
    Chuyển logbook của DEAP thành pandas.DataFrame có cấu trúc phẳng
    và đổi tên cột cho dễ đọc, sắp xếp theo yêu cầu.
    """
    if not logbook:
        return pd.DataFrame()

    records = []
    # Map original DEAP chapter names to desired prefixes
    prefix_map = {
        'obj1': 'coverage',
        'obj2': 'altitude',
        'obj3': 'costs' # 'costs' will be mapped to 'num_sats' in final output order
    }
    # Map DEAP stat suffixes to desired suffixes
    stat_suffix_map = {
        'avg': 'mean',
        'min': 'min',
        'max': 'max',
        'std': 'std'
    }

    # Process each generation's record
    for i, gen_record in enumerate(logbook):
        current_gen_data = {}

        # Handle base columns
        if 'gen' in gen_record:
            current_gen_data['generation'] = gen_record['gen']
        if 'nevals' in gen_record:
            current_gen_data['nevals'] = gen_record['nevals']
        if 'new_hof' in gen_record:
            current_gen_data['new_hof'] = gen_record['new_hof']

        # Handle chapter data (objective statistics)
        if hasattr(logbook, "chapters") and logbook.chapters:
            for chapter_name, chapter_data_list in logbook.chapters.items():
                if chapter_name in prefix_map and i < len(chapter_data_list):
                    prefix = prefix_map[chapter_name]
                    chapter_record = chapter_data_list[i]
                    for stat_key, stat_value in chapter_record.items():
                        # stat_key could be 'avg', 'min', 'max', 'std'
                        new_stat_suffix = stat_suffix_map.get(stat_key, stat_key)
                        current_gen_data[f"{prefix}_{new_stat_suffix}"] = stat_value
        else: # Fallback if logbook doesn't have chapters but has stats directly in gen_record
            for key, value in gen_record.items():
                if key not in ['gen', 'nevals', 'new_hof']: # Avoid re-processing base columns
                    matched_obj = False
                    for obj_name, prefix in prefix_map.items():
                        for old_stat_suffix, new_stat_suffix in stat_suffix_map.items():
                            if key == f"{obj_name}{old_stat_suffix}":
                                current_gen_data[f"{prefix}_{new_stat_suffix}"] = value
                                matched_obj = True
                                break
                        if matched_obj:
                            break
                    if not matched_obj: # If it's another unmapped key, just add it
                        current_gen_data[key] = value

        records.append(current_gen_data)

    df_final = pd.DataFrame(records)

    # Define desired order of columns
    objective_order = ['coverage', 'altitude', 'costs']
    stat_order = ['max', 'min', 'mean', 'std'] # Corrected to use 'mean' consistently

    ordered_columns_list = []
    # Add base columns first
    for col in ['generation', 'nevals', 'new_hof']:
        if col in df_final.columns:
            ordered_columns_list.append(col)

    # Add objective-specific columns in desired order
    for obj_prefix in objective_order:
        for stat_suffix in stat_order:
            col_name = f"{obj_prefix}_{stat_suffix}" # Now using underscore consistently
            if col_name in df_final.columns:
                ordered_columns_list.append(col_name)

    # Add any other columns not explicitly listed (for robustness)
    for col in df_final.columns:
        if col not in ordered_columns_list:
            ordered_columns_list.append(col)

    # Filter and reorder the DataFrame
    df_final = df_final[ordered_columns_list]

    return df_final

# Create a dictionary to store logbook dataframes from each checkpoint
logbook_dataframes = {}

# Iterate through the loaded checkpoints data
for checkpoint_key, data in loaded_checkpoints_data.items(): # Iterate using the extracted integer key
    logbook = data["logbook"]
    # Convert the logbook to a dataframe and store it in the dictionary
    # Use the checkpoint_key (which is the extracted integer) as the dictionary key
    logbook_dataframes[checkpoint_key] = logbook_to_dataframe(logbook)

# Now logbook_dataframes is a dictionary where keys are checkpoint file paths
# and values are the corresponding logbook dataframes.
# You can access a logbook dataframe like: logbook_dataframes['/content/drive/MyDrive/PTNK/NCKH/Sprint 11/checkpoint_constraint_0.pkl']
# You can display the first logbook dataframe like:
# display(logbook_dataframes[CHECKPOINT_FILE[0]])

# Display all logbook dataframes
for key, df in logbook_dataframes.items():
    print(f"Logbook for checkpoint key: {key}")
    display(df)

Logbook for checkpoint key: 0


,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,...,costs_std,coverage_gen,coverage_nevals,coverage_new_hof,altitude_gen,altitude_nevals,altitude_new_hof,costs_gen,costs_nevals,costs_new_hof
0,0,50,33,1.0,0.009986,0.652314,0.341445,993.0,513.0,760.64,...,64.505194,0,50,33,0,50,33,0,50,33
1,1,50,26,1.0,0.009986,0.649202,0.313419,988.0,513.0,721.50,...,53.737882,1,50,26,1,50,26,1,50,26
2,2,50,26,1.0,0.009986,0.602706,0.330290,988.0,513.0,704.82,...,58.006538,2,50,26,2,50,26,2,50,26
3,3,50,28,1.0,0.009986,0.593248,0.335608,988.0,513.0,703.34,...,56.554133,3,50,28,3,50,28,3,50,28
4,4,50,26,1.0,0.005918,0.602321,0.339599,988.0,513.0,711.84,...,55.890021,4,50,26,4,50,26,4,50,26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,50,3,1.0,0.003144,0.625546,0.373286,996.0,500.0,710.00,...,55.000160,96,50,3,96,50,3,96,50,3
97,97,50,6,1.0,0.003144,0.647299,0.352850,996.0,500.0,704.92,...,53.918472,97,50,6,97,50,6,97,50,6
98,98,50,7,1.0,0.003144,0.631209,0.351195,996.0,500.0,705.88,...,54.380588,98,50,7,98,50,7,98,50,7
99,99,50,12,1.0,0.003144,0.674733,0.344498,996.0,500.0,729.88,...,54.059597,99,50,12,99,50,12,99,50,12


Logbook for checkpoint key: 50


,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,...,costs_std,coverage_gen,coverage_nevals,coverage_new_hof,altitude_gen,altitude_nevals,altitude_new_hof,costs_gen,costs_nevals,costs_new_hof
0,0,50,33,1.0,0.009986,0.652314,0.341445,993.0,513.0,760.64,...,64.505194,0,50,33,0,50,33,0,50,33
1,1,50,24,1.0,0.545354,0.887808,0.121357,993.0,513.0,746.76,...,60.894663,1,50,24,1,50,24,1,50,24
2,2,50,22,1.0,0.545354,0.874539,0.123364,993.0,513.0,733.28,...,47.644622,2,50,22,2,50,22,2,50,22
3,3,50,21,1.0,0.561997,0.858787,0.128955,984.0,513.0,713.84,...,55.795086,3,50,21,3,50,21,3,50,21
4,4,50,19,1.0,0.532963,0.850295,0.125861,989.0,513.0,706.00,...,47.918531,4,50,19,4,50,19,4,50,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,50,15,1.0,0.500693,0.801990,0.177530,974.0,501.0,667.26,...,49.792369,96,50,15,96,50,15,96,50,15
97,97,50,13,1.0,0.500693,0.795543,0.183443,974.0,501.0,672.28,...,50.171849,97,50,13,97,50,13,97,50,13
98,98,50,9,1.0,0.500693,0.813920,0.178949,974.0,501.0,689.32,...,49.609660,98,50,9,98,50,9,98,50,9
99,99,50,7,1.0,0.500693,0.804496,0.184523,979.0,501.0,693.86,...,50.298807,99,50,7,99,50,7,99,50,7


Logbook for checkpoint key: 60


...

,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,...,costs_std,coverage_gen,coverage_nevals,coverage_new_hof,altitude_gen,altitude_nevals,altitude_new_hof,costs_gen,costs_nevals,costs_new_hof
0,0,50,33,1.0,0.009986,0.652314,0.341445,993.0,513.0,760.64,...,64.505194,0,50,33,0,50,33,0,50,33
1,1,50,22,1.0,0.833934,0.971310,0.048442,993.0,513.0,792.96,...,73.015659,1,50,22,1,50,22,1,50,22
2,2,50,16,1.0,0.990476,0.998606,0.002629,988.0,513.0,800.70,...,63.881481,2,50,16,2,50,16,2,50,16
3,3,50,9,1.0,0.990014,0.998034,0.002958,988.0,513.0,767.92,...,57.883801,3,50,9,3,50,9,3,50,9
4,4,50,12,1.0,0.990014,0.997764,0.003202,988.0,513.0,735.26,...,58.595713,4,50,12,4,50,12,4,50,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,50,2,1.0,0.990476,0.997341,0.003056,980.0,501.0,677.62,...,37.187767,96,50,2,96,50,2,96,50,2
97,97,50,7,1.0,0.990476,0.997187,0.003133,980.0,501.0,669.78,...,37.200973,97,50,7,97,50,7,97,50,7
98,98,50,6,1.0,0.990476,0.997150,0.003121,980.0,501.0,681.20,...,37.915849,98,50,6,98,50,6,98,50,6
99,99,50,3,1.0,0.990476,0.997246,0.003033,980.0,501.0,680.08,...,38.270725,99,50,3,99,50,3,99,50,3


Logbook for checkpoint key: 100


,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,...,costs_std,coverage_gen,coverage_nevals,coverage_new_hof,altitude_gen,altitude_nevals,altitude_new_hof,costs_gen,costs_nevals,costs_new_hof
0,0,50,33,1.0,0.009986,0.652314,0.341445,993.0,513.0,760.64,...,64.505194,0,50,33,0,50,33,0,50,33
1,1,50,21,1.0,0.833934,0.971626,0.050178,993.0,513.0,790.68,...,72.686934,1,50,21,1,50,21,1,50,21
2,2,50,17,1.0,0.998058,0.999784,0.000479,993.0,513.0,802.42,...,65.820608,2,50,17,2,50,17,2,50,17
3,3,50,10,1.0,1.000000,1.000000,0.000000,991.0,513.0,778.94,...,64.882836,3,50,10,3,50,10,3,50,10
4,4,50,5,1.0,1.000000,1.000000,0.000000,991.0,513.0,731.64,...,65.534248,4,50,5,4,50,5,4,50,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,50,1,1.0,1.000000,1.000000,0.000000,854.0,511.0,757.16,...,22.178043,96,50,1,96,50,1,96,50,1
97,97,50,4,1.0,1.000000,1.000000,0.000000,854.0,511.0,757.16,...,22.178043,97,50,4,97,50,4,97,50,4
98,98,50,2,1.0,1.000000,1.000000,0.000000,854.0,511.0,757.16,...,22.178043,98,50,2,98,50,2,98,50,2
99,99,50,3,1.0,1.000000,1.000000,0.000000,854.0,511.0,757.16,...,22.178043,99,50,3,99,50,3,99,50,3


Logbook for checkpoint key: 2016


,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,coverage_gen,coverage_nevals,coverage_new_hof,altitude_gen,altitude_nevals,altitude_new_hof
0,0,50,1,0.944523,0.441840,0.827724,0.139911,1.0,0.457513,0.880577,0.157296,0,50,1,0,50,1
1,1,50,1,0.945381,0.534494,0.894852,0.075263,1.0,0.545354,0.955577,0.085813,1,50,1,1,50,1
2,2,50,1,0.936706,0.296069,0.875379,0.116858,1.0,0.284882,0.936423,0.133498,2,50,1,2,50,1
3,3,50,0,0.936748,0.739759,0.912763,0.047636,1.0,0.777901,0.979197,0.054339,3,50,0,3,50,0
4,4,50,0,0.939559,0.848248,0.926618,0.016062,1.0,0.912159,0.994602,0.017426,4,50,0,4,50,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,50,0,0.942516,0.714318,0.900267,0.055388,1.0,0.761073,0.963375,0.063980,96,50,0,96,50,0
97,97,50,0,0.949290,0.714318,0.909825,0.041919,1.0,0.761073,0.975077,0.048417,97,50,0,97,50,0
98,98,50,0,0.942516,0.493916,0.881601,0.091251,1.0,0.493111,0.942983,0.104467,98,50,0,98,50,0
99,99,50,0,0.938612,0.592904,0.890454,0.085703,1.0,0.616736,0.953570,0.096605,99,50,0,99,50,0


In [53]:
# @title
def reprocess_logbook_for_roullete(pop_history):
    """
    Reprocesses the logbook-like data for the roullete checkpoint (key 2016)
    to correctly calculate statistics based on pop_history for the entire population.

    Parameters
    ----------
    pop_history : list
        The pop_history list loaded from the roullete checkpoint.
        Each element is a dictionary with 'gen', 'population', and 'fitnesses'.

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing the reprocessed statistics per generation,
        matching the format of logbook_to_dataframe.
    """
    reprocessed_data = []

    for gen_data in tqdm(pop_history, desc="Reprocessing Roullete Stats from Pop History"):
        generation_num = gen_data['gen'] # Get generation number
        population_genes = gen_data['population']
        fitnesses = gen_data['fitnesses']

        if not fitnesses:
            print(f"Warning: No fitness data found for generation {generation_num}. Skipping.")
            continue

        # Extract fitness values for each objective
        coverages = [fit[0] for fit in fitnesses]
        altitudes = [fit[1] for fit in fitnesses] # Altitude is the second fitness value
        costs = [fit[2] for fit in fitnesses]     # Costs (num_sats) is the third fitness value

        # Calculate statistics for the current generation's population
        reprocessed_row = {
            'generation': generation_num,
            'nevals': len(fitnesses), # Number of evaluated individuals in this population
            'new_hof': 0, # Placeholder, as this function doesn't track new HOF additions directly from pop_history

            'coverage_max': np.max(coverages),
            'coverage_min': np.min(coverages),
            'coverage_mean': np.mean(coverages),
            'coverage_std': np.std(coverages),

            'altitude_max': np.max(altitudes),
            'altitude_min': np.min(altitudes),
            'altitude_mean': np.mean(altitudes),
            'altitude_std': np.std(altitudes),

            'costs_max': np.max(costs),
            'costs_min': np.min(costs),
            'costs_mean': np.mean(costs),
            'costs_std': np.std(costs),
        }
        reprocessed_data.append(reprocessed_row)

    df_final = pd.DataFrame(reprocessed_data)

    # Define desired order of columns explicitly to match logbook_to_dataframe
    desired_order = [
        'generation', 'nevals', 'new_hof',
        'coverage_max', 'coverage_min', 'coverage_mean', 'coverage_std',
        'altitude_max', 'altitude_min', 'altitude_mean', 'altitude_std',
        'costs_max', 'costs_min', 'costs_mean', 'costs_std'
    ]

    # Ensure all desired columns exist, if not, add them with NaN
    for col in desired_order:
        if col not in df_final.columns:
            df_final[col] = np.nan # Use NaN for missing data

    # Reorder the DataFrame columns
    df_final = df_final[desired_order]

    return df_final

# Assuming toolbox.individualCreator is the correct function to recreate individuals
# and loaded_checkpoints_data is already loaded

# Check if the roullete checkpoint data exists and has pop_history
roullete_key = 2016
if roullete_key in loaded_checkpoints_data and isinstance(loaded_checkpoints_data[roullete_key].get('pop_history'), list):
    print(f"Reprocessing stats for checkpoint key: {roullete_key} from pop_history")
    roullete_data = loaded_checkpoints_data[roullete_key]
    roullete_pop_history = roullete_data['pop_history']

    # Reprocess the logbook for the roullete checkpoint based on pop_history
    reprocessed_roullete_logbook_df = reprocess_logbook_for_roullete(
        roullete_pop_history
    )

    # Update the logbook_dataframes dictionary with the reprocessed data
    # Ensure logbook_dataframes dictionary is initialized if it doesn't exist
    if 'logbook_dataframes' not in locals():
        logbook_dataframes = {}
    logbook_dataframes[roullete_key] = reprocessed_roullete_logbook_df

    print(f"✅ Đã xử lý lại dữ liệu thống kê cho checkpoint {roullete_key} từ pop_history.")
    display(logbook_dataframes[roullete_key].head()) # Display head of the reprocessed data
else:
    print(f"⚠️ Dữ liệu pop_history cho checkpoint {roullete_key} không được tìm thấy hoặc không đúng định dạng. Bỏ qua xử lý lại thống kê.")

Reprocessing stats for checkpoint key: 2016 from pop_history


Reprocessing Roullete Stats from Pop History: 100%|██████████| 101/101 [00:00<00:00, 5111.12it/s]

✅ Đã xử lý lại dữ liệu thống kê cho checkpoint 2016 từ pop_history.


,generation,nevals,new_hof,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,costs_max,costs_min,costs_mean,costs_std
0,0,50,0,1.0,0.457513,0.880577,0.157296,993.0,513.0,748.64,134.389845,286.0,24.0,117.12,70.014181
1,1,50,0,1.0,0.545354,0.955577,0.085813,993.0,547.0,751.32,134.756883,286.0,51.0,162.48,72.022008
2,2,50,0,1.0,0.284882,0.936423,0.133498,993.0,547.0,800.82,126.463543,288.0,16.0,160.96,76.085993
3,3,50,0,1.0,0.777901,0.979197,0.054339,993.0,552.0,819.38,108.527027,286.0,48.0,180.76,63.853758
4,4,50,0,1.0,0.912159,0.994602,0.017426,993.0,552.0,817.56,103.974643,286.0,54.0,187.20,63.128757


In [27]:
# @title
import pandas as pd
import os

def export_dataframes_to_csv(data_structure, folder, prefix="", is_dict=True):
    """
    Exports pandas DataFrames to CSV files.

    Parameters
    ----------
    data_structure : dict or pandas.DataFrame
        A dictionary of DataFrames or a single DataFrame to export.
    folder : str
        The output folder path.
    prefix : str
        A prefix to prepend to the filenames.
    is_dict : bool
        True if data_structure is a dictionary of DataFrames, False if it's a single DataFrame.
    """
    os.makedirs(folder, exist_ok=True)

    if is_dict:
        for key, df in data_structure.items():
            if isinstance(df, pd.DataFrame) and not df.empty:
                filename = os.path.join(folder, f"{prefix}_{key}.csv")
                df.to_csv(filename, index=True)
                print(f"✅ Exported {prefix}_{key}.csv")
            else:
                print(f"⚠️ Skipping {prefix}_{key}: Not a DataFrame or is empty.")
    else:
        if isinstance(data_structure, pd.DataFrame) and not data_structure.empty:
            filename = os.path.join(folder, f"{prefix}.csv")
            data_structure.to_csv(filename, index=True)
            print(f"✅ Exported {prefix}.csv")
        else:
            print(f"⚠️ Skipping {prefix}: Not a DataFrame or is empty.")


# if 'logbook_dataframes' in locals() and logbook_dataframes:
#     print("--- Exporting logbook_dataframes ---")
#     export_dataframes_to_csv(logbook_dataframes, "/content/vietnam-satellite-simulation/logbook_stats", "logbook_stats")
# else:
#     print("⚠️ `logbook_dataframes` not found or is empty. Cannot export.")

In [ ]:
# @title
import matplotlib.pyplot as plt
import os
import re # Import re for parsing filenames

def plot_comparison_across_checkpoints(logbook_dataframes, loaded_checkpoints_data, output_folder):
    """
    Generates plots comparing the evolution of each statistic (avg, std)
    for each objective (coverage, altitude, costs) across all loaded logbooks.

    Parameters
    ----------
    logbook_dataframes : dict
        A dictionary where keys are checkpoint integers and values are the
        corresponding logbook dataframes (as created by logbook_to_dataframe).
    loaded_checkpoints_data : dict
        A dictionary where keys are checkpoint integers and values are the
        full loaded data, including the original file path.
    output_folder : str
        The path to the folder where the plots should be saved.
    """
    if not logbook_dataframes:
        print("Không có dữ liệu logbook để vẽ biểu đồ.")
        return

    objectives = ['coverage', 'altitude', 'costs']
    statistics = ['avg', 'std']

    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # MỚI: Kiểm tra CUSTOM_LABELS và sắp xếp keys cho nhất quán
    use_custom_labels = False
    sorted_checkpoint_keys = sorted(logbook_dataframes.keys()) # Keys for CUSTOM_LABELS mapping
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(sorted_checkpoint_keys):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(sorted_checkpoint_keys)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # Define markers and linestyles for plot differentiation
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']

    # Iterate through each objective to create comparison plots for its statistics
    for obj_name in objectives:
        print(f"\nGenerating comparison plots for objective: {obj_name.replace('_', ' ').title()}")
        for stat_name in statistics:
            plt.figure(figsize=(12, 7))
            title = f"Evolution of {stat_name.title()} {obj_name.replace('_', ' ').title()} across Checkpoints"
            ylabel = f"{obj_name.replace('_', ' ').title()} ({stat_name.title()})"

            plotted_any = False
            # Iterate through all checkpoints to plot their data on the same figure
            # MỚI: Dùng enumerate để lấy index (i) cho markers/linestyles
            for i, checkpoint_key in enumerate(sorted_checkpoint_keys): # Iterate using sorted keys
                current_logbook_df = logbook_dataframes[checkpoint_key] # Get DF using the key

                col_name = f'{obj_name}_{stat_name}'

                if col_name in current_logbook_df.columns:
                    # MỚI: Quyết định label
                    if use_custom_labels:
                        label_name = CUSTOM_LABELS[i]
                    else:
                        label_name = f"Test {checkpoint_key}" # Fallback

                    # MỚI: Use dynamic markers and linestyles
                    marker = markers[i % len(markers)]
                    linestyle = linestyles[i % len(linestyles)]

                    plt.plot(current_logbook_df['gen'], current_logbook_df[col_name],
                             label=label_name, # MỚI: Sử dụng label_name đã định nghĩa
                             marker=marker,
                             linestyle=linestyle,
                             markersize=5) # MỚI: Thêm markersize
                    plotted_any = True
                else:
                    print(f"Cột '{col_name}' không tìm thấy trong logbook từ Test {checkpoint_key}. Bỏ qua plotting cho cột này.")


            if plotted_any:
                plt.gca().spines[['top', 'right',]].set_visible(False) # MỚI: Thêm style
                plt.xlabel("Generation")
                plt.ylabel(ylabel)
                plt.title(title)
                plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left') # MỚI: Thêm style legend
                plt.grid(True)

                # Define the filename based on the objective and statistic
                filename = f"comparison_{stat_name}_{obj_name}_plot.png" # MỚI: Đổi tên file để phân biệt avg/std
                filepath = os.path.join(output_folder, filename)

                # Save the figure
                plt.savefig(filepath, bbox_inches='tight') # MỚI: Thêm bbox_inches
                print(f"✅ Saved comparison plot to {filepath}")
                plt.close() # Close the figure to free up memory
            else:
                 print(f"Không có dữ liệu để vẽ biểu đồ cho {obj_name} - {stat_name}. Bỏ qua tạo plot.")


# Example usage (assuming logbook_dataframes and OUTPUT_FOLDER are defined in previous cells):
# Pass loaded_checkpoints_data to the function
plot_comparison_across_checkpoints(logbook_dataframes, loaded_checkpoints_data, OUTPUT_FOLDER)

Biểu đồ sau cho thấy quá trình hội tụ của từng mục tiêu qua các thế hệ, giúp đánh giá tốc độ cải thiện của quần thể. Nhờ biểu đồ này, ta có thể nhận thấy sự ổn định và hội tụ của quần thể khi thuật toán tiến hoá dần đạt tới vùng Pareto tối ưu.

### **8.2. Sự Tiến hóa của các Lời giải Ưu tú (Hall of Fame History)**
Hall of Fame (HOF) lưu giữ các cá thể tốt nhất được phát hiện trong mỗi thế hệ. Để phân tích quá trình cải thiện của các lời giải ưu tú này, toàn bộ lịch sử HOF được lưu thành DataFrame, trong đó mỗi hàng thể hiện:

*   Các giá trị fitness của cá thể tốt nhất ở từng thế hệ
*   Thông tin vị trí hoặc biến thiết kế của cá thể tương ứng

In [45]:
# @title
import pandas as pd
import re # Import regex for filename parsing

# Create a dictionary to hold the history_hall_of_fame_df DataFrames for all checkpoints
# The key will be the checkpoint key (integer)
all_history_hall_of_fame_dfs = {}

# Iterate through the loaded checkpoints data
if 'loaded_checkpoints_data' not in locals() or not loaded_checkpoints_data:
    print("Không có dữ liệu checkpoint nào được tải. Vui lòng chạy cell tải checkpoint trước.")
else:
    # Sort the checkpoint keys (integers) to process them in order
    sorted_checkpoint_keys = sorted(loaded_checkpoints_data.keys())

    for checkpoint_key in sorted_checkpoint_keys:
        data = loaded_checkpoints_data[checkpoint_key]
        halloffame_history = data['halloffame_history']
        logbook = data['logbook'] # We might need the logbook to get 'new_hof'

        # Find the original file path string from the CHECKPOINT_FILE list using the checkpoint key
        original_file_path = f"Checkpoint_{checkpoint_key}" # Default name
        for cp_file in CHECKPOINT_FILE:
             match = re.search(r'_(\d+)\.pkl$', os.path.basename(cp_file))
             if match and int(match.group(1)) == checkpoint_key:
                 original_file_path = cp_file
                 break


        # Create a list to hold the data for the DataFrame for the current checkpoint
        history_hall_of_fame_data = []

        # Iterate through the history of the hall of fame (list of dictionaries) for this checkpoint
        for hof_snapshot in halloffame_history:
            gen = hof_snapshot["gen"]
            halloffame_list = hof_snapshot["halloffame"]
            fitnesses_list = hof_snapshot["fitnesses"]

            # Find the corresponding 'new_hof' value for this generation from the logbook
            # Assuming the logbook is a list of dictionaries and 'gen' is a key
            new_hof_value = 0
            for record in logbook:
                if record.get('gen') == gen:
                     # Assuming 'new_hof' is a key in the logbook records
                    new_hof_value = record.get('new_hof', 0)
                    break


            # Iterate through each individual in the hall of fame for this generation
            for i in range(len(halloffame_list)):
                individual = halloffame_list[i]
                fitness_values = fitnesses_list[i]
                history_hall_of_fame_data.append({
                    # "Checkpoint_Key": checkpoint_key, # Add checkpoint key as a column
                    # "Checkpoint_File": os.path.basename(original_file_path), # Add checkpoint file name as a column
                    "generation": gen,
                    "index": i + 1,
                    "coverage": fitness_values[0],
                    # "New Individuals in Hall of Fame": new_hof_value, # Add new_hof column
                    "altitude": individual[0],
                    "costs": fitness_values[2],
                    "inclination": individual[1],
                    "num_planes": int(individual[2]),
                    "num_sats_per_plane": int(individual[3]),
                    "phasing": individual[4],
                    # "Fitness (Altitude)": fitness_values[1],
                })

        # Create the pandas DataFrame for the current checkpoint's history
        history_hall_of_fame_df = pd.DataFrame(history_hall_of_fame_data)

        # Store the resulting DataFrame in the dictionary with the checkpoint key
        all_history_hall_of_fame_dfs[checkpoint_key] = history_hall_of_fame_df


# Display the dictionary of DataFrames (optional)
print("Dictionary of history_hall_of_fame_df DataFrames:")
for key, df in all_history_hall_of_fame_dfs.items():
    print(f"\nDataFrame from checkpoint key {key}:")
    display(df)

# There is no combined_df needed at this level based on the user's request to use a dictionary.
# The subsequent cells that use all_history_hall_of_fame_dfs will need to be updated
# to iterate through the dictionary instead of a list.

Dictionary of history_hall_of_fame_df DataFrames:

DataFrame from checkpoint key 0:


,generation,index,coverage,altitude,costs,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,1.000000,691,135.0,22,9,15,2
1,0,2,1.000000,882,133.0,24,7,19,4
2,0,3,0.999075,852,88.0,23,4,22,2
3,0,4,0.998890,552,216.0,25,12,18,2
4,0,5,0.992880,805,91.0,16,7,13,5
...,...,...,...,...,...,...,...,...,...
60339,100,956,0.003883,563,1.0,21,1,1,0
60340,100,957,0.003606,550,1.0,20,1,1,0
60341,100,958,0.003514,538,1.0,21,1,1,0
60342,100,959,0.003236,529,1.0,21,1,1,0



DataFrame from checkpoint key 50:


,generation,index,coverage,altitude,costs,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,1.000000,691,135.0,22,9,15,2
1,0,2,1.000000,882,133.0,24,7,19,4
2,0,3,0.999075,852,88.0,23,4,22,2
3,0,4,0.998890,552,216.0,25,12,18,2
4,0,5,0.992880,805,91.0,16,7,13,5
...,...,...,...,...,...,...,...,...,...
45244,100,768,0.048636,805,2.0,19,2,1,1
45245,100,769,0.045123,650,3.0,18,3,1,2
45246,100,770,0.042534,769,2.0,15,2,1,1
45247,100,771,0.009986,802,1.0,21,1,1,1



DataFrame from checkpoint key 60:


...

,generation,index,coverage,altitude,costs,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,1.000000,691,135.0,22,9,15,2
1,0,2,1.000000,882,133.0,24,7,19,4
2,0,3,0.999075,852,88.0,23,4,22,2
3,0,4,0.998890,552,216.0,25,12,18,2
4,0,5,0.992880,805,91.0,16,7,13,5
...,...,...,...,...,...,...,...,...,...
22348,100,365,0.067684,796,3.0,16,3,1,1
22349,100,366,0.064448,502,23.0,20,1,23,0
22350,100,367,0.052427,650,4.0,19,4,1,3
22351,100,368,0.030791,602,2.0,21,2,1,1



DataFrame from checkpoint key 100:


,generation,index,coverage,altitude,costs,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,1.000000,691,135.0,22,9,15,2
1,0,2,1.000000,882,133.0,24,7,19,4
2,0,3,0.999075,852,88.0,23,4,22,2
3,0,4,0.998890,552,216.0,25,12,18,2
4,0,5,0.992880,805,91.0,16,7,13,5
...,...,...,...,...,...,...,...,...,...
15726,100,231,0.045677,772,2.0,19,2,1,1
15727,100,232,0.020620,665,4.0,22,1,4,1
15728,100,233,0.018031,502,7.0,19,1,7,0
15729,100,234,0.017753,607,5.0,19,1,5,0



DataFrame from checkpoint key 2016:


,generation,index,coverage,altitude,costs,inclination,num_planes,num_sats_per_plane,phasing
0,0,1,1.000000,691.0,135.0,22.0,9,15,2.0
1,0,2,1.000000,882.0,133.0,24.0,7,19,4.0
2,0,3,0.998890,552.0,216.0,25.0,12,18,2.0
3,0,4,0.992880,805.0,91.0,16.0,7,13,5.0
4,0,5,0.981322,739.0,90.0,23.0,5,18,1.0
...,...,...,...,...,...,...,...,...,...
8121,100,117,0.281923,769.0,14.0,15.0,14,1,1.0
8122,100,118,0.268978,924.0,10.0,21.0,10,1,8.0
8123,100,119,0.255294,526.0,24.0,21.0,3,8,1.0
8124,100,120,0.235969,891.0,10.0,24.0,5,2,4.0


In [50]:
# @title
import pandas as pd

# Ensure all_history_hall_of_fame_dfs is available
if 'all_history_hall_of_fame_dfs' not in locals() or not all_history_hall_of_fame_dfs:
    print("Danh sách all_history_hall_of_fame_dfs không tồn tại hoặc rỗng. Vui lòng chạy cell trước để tạo danh sách này.")
else:
    # Create a dictionary to hold the hall_of_fame_gene_stats DataFrames for all checkpoints
    # The key will be the checkpoint key (integer)
    all_hall_of_fame_gene_stats_dfs = {}

    for checkpoint_key, history_hall_of_fame_df in all_history_hall_of_fame_dfs.items():

        if history_hall_of_fame_df.empty:
             print(f"⚠️ DataFrame Hall of Fame cho checkpoint key {checkpoint_key} rỗng. Bỏ qua DataFrame này.")
             continue

        # Define the fitness columns for which to calculate statistics
        fitness_columns = [
            "coverage",
            "altitude",
            "costs"
        ]

        # Group by 'generation' and calculate statistics for all fitness columns
        # Only include columns that exist in the current DataFrame
        existing_fitness_columns = [col for col in history_hall_of_fame_df.columns if col in fitness_columns]
        hall_of_fame_gene_stats = history_hall_of_fame_df.groupby(['generation'])[existing_fitness_columns].agg(['max', 'min', 'mean', 'std'])

        # Flatten the multi-level columns and rename 'costs' to 'num_sats'
        new_columns = []
        for col_name, stat in hall_of_fame_gene_stats.columns.values:
            if col_name == 'costs':
                new_columns.append(f"num_sats_{stat}")
            else:
                new_columns.append(f"{col_name}_{stat}")
        hall_of_fame_gene_stats.columns = new_columns

        # Reset index to turn 'generation' into a column if it's the index
        hall_of_fame_gene_stats = hall_of_fame_gene_stats.reset_index()

        # Define desired column order
        desired_order = [
            'generation',
            'coverage_max', 'coverage_min', 'coverage_mean', 'coverage_std',
            'altitude_max', 'altitude_min', 'altitude_mean', 'altitude_std',
            'num_sats_max', 'num_sats_min', 'num_sats_mean', 'num_sats_std'
        ]

        # Filter and reorder columns that actually exist in the DataFrame
        final_columns = [col for col in desired_order if col in hall_of_fame_gene_stats.columns]
        hall_of_fame_gene_stats = hall_of_fame_gene_stats[final_columns]

        # Store the resulting DataFrame in the dictionary with the checkpoint key
        all_hall_of_fame_gene_stats_dfs[checkpoint_key] = hall_of_fame_gene_stats

    # Display the resulting dictionary of DataFrames (optional)
    print("Dictionary of hall_of_fame_gene_stats DataFrames:")
    if all_hall_of_fame_gene_stats_dfs:
        for key, df in all_hall_of_fame_gene_stats_dfs.items():
            print(f"\nDataFrame from checkpoint key {key}:\n") # Added \n for better output
            display(df)

    else:
        print("Không có hall_of_fame_gene_stats nào được tạo.")

Dictionary of hall_of_fame_gene_stats DataFrames:

DataFrame from checkpoint key 0:



,generation,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,num_sats_max,num_sats_min,num_sats_mean,num_sats_std
0,0,1.0,0.009986,0.629967,0.333329,984,513,703.030303,129.730848,216.0,1.0,59.000000,47.524993
1,1,1.0,0.009986,0.656468,0.317879,988,513,718.392857,145.392094,286.0,1.0,60.964286,52.412339
2,2,1.0,0.009986,0.631926,0.319366,988,513,700.133333,141.183447,286.0,1.0,58.506667,51.289421
3,3,1.0,0.009986,0.630174,0.304146,988,513,700.957447,141.544546,286.0,1.0,54.127660,46.325690
4,4,1.0,0.005918,0.604624,0.315193,988,513,699.911504,140.687506,286.0,1.0,50.336283,44.323408
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1.0,0.003144,0.507201,0.308666,1000,500,687.229456,142.558744,180.0,1.0,35.182497,30.675093
97,97,1.0,0.003144,0.507133,0.308844,1000,500,687.836170,142.764695,180.0,1.0,35.161702,30.655415
98,98,1.0,0.003144,0.507365,0.308701,1000,500,688.218816,142.762628,180.0,1.0,35.137421,30.601209
99,99,1.0,0.003144,0.506941,0.308127,1000,500,688.650628,142.937438,180.0,1.0,35.018828,30.471163



DataFrame from checkpoint key 50:



,generation,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,num_sats_max,num_sats_min,num_sats_mean,num_sats_std
0,0,1.0,0.009986,0.629967,0.333329,984,513,703.030303,129.730848,216.0,1.0,59.000000,47.524993
1,1,1.0,0.009986,0.641796,0.333616,993,513,713.826923,136.706602,276.0,1.0,59.730769,53.281758
2,2,1.0,0.009986,0.679477,0.315888,984,513,713.939394,134.398933,276.0,1.0,61.181818,50.153276
3,3,1.0,0.009986,0.691053,0.301469,984,513,699.415584,132.404916,276.0,1.0,64.194805,54.261317
4,4,1.0,0.009986,0.709132,0.291417,989,513,698.647727,131.960116,276.0,1.0,62.193182,46.379707
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1.0,0.007305,0.638028,0.276305,994,500,677.997315,144.456722,198.0,1.0,47.832215,35.272519
97,97,1.0,0.007305,0.638049,0.275316,994,500,677.898667,144.579305,198.0,1.0,47.753333,35.166348
98,98,1.0,0.007305,0.638036,0.274577,994,500,677.562582,144.140684,198.0,1.0,47.708827,35.041488
99,99,1.0,0.007305,0.638441,0.274188,994,500,677.820446,143.961811,198.0,1.0,47.707733,35.018777



DataFrame from checkpoint key 60:



...

,generation,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,num_sats_max,num_sats_min,num_sats_mean,num_sats_std
0,0,1.0,0.009986,0.629967,0.333329,984,513,703.030303,129.730848,216.0,1.0,59.000000,47.524993
1,1,1.0,0.009986,0.658974,0.331072,988,513,705.060000,135.475039,276.0,1.0,64.480000,57.555028
2,2,1.0,0.009986,0.690739,0.327687,988,513,703.721311,132.508758,276.0,1.0,68.409836,57.049796
3,3,1.0,0.009986,0.675069,0.322153,988,513,707.380952,130.556925,276.0,1.0,63.873016,52.862418
4,4,1.0,0.009986,0.705551,0.315696,989,513,706.873239,134.223367,276.0,1.0,67.915493,54.208525
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1.0,0.010818,0.734134,0.284536,993,500,663.562674,133.186918,187.0,1.0,61.537604,36.826304
97,97,1.0,0.010818,0.728252,0.286829,993,500,662.487465,133.072603,187.0,1.0,60.916435,36.961685
98,98,1.0,0.010818,0.728568,0.285785,993,500,662.469780,133.682600,187.0,1.0,60.892857,36.738478
99,99,1.0,0.010818,0.728932,0.285215,993,500,662.773224,133.520284,187.0,1.0,60.838798,36.671944



DataFrame from checkpoint key 100:



,generation,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,num_sats_max,num_sats_min,num_sats_mean,num_sats_std
0,0,1.0,0.009986,0.629967,0.333329,984,513,703.030303,129.730848,216.0,1.0,59.000000,47.524993
1,1,1.0,0.009986,0.649500,0.328909,988,513,706.978723,134.852172,276.0,1.0,61.531915,55.367488
2,2,1.0,0.009986,0.692751,0.315249,988,513,698.033898,129.762378,276.0,1.0,66.322034,52.752657
3,3,1.0,0.009986,0.724688,0.312117,988,513,694.469697,126.828075,276.0,1.0,74.924242,59.332002
4,4,1.0,0.009986,0.737111,0.306294,989,513,693.366197,130.594054,276.0,1.0,79.718310,65.265870
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1.0,0.009986,0.654569,0.307987,995,500,675.668142,131.333335,180.0,1.0,54.681416,39.826572
97,97,1.0,0.009986,0.652106,0.308505,995,500,676.965217,130.837973,180.0,1.0,54.300000,39.737254
98,98,1.0,0.009986,0.654247,0.308130,995,500,676.771552,131.017644,192.0,1.0,54.840517,40.593385
99,99,1.0,0.009986,0.654479,0.307470,995,500,677.081545,130.771310,192.0,1.0,54.742489,40.533434



DataFrame from checkpoint key 2016:



,generation,coverage_max,coverage_min,coverage_mean,coverage_std,altitude_max,altitude_min,altitude_mean,altitude_std,num_sats_max,num_sats_min,num_sats_mean,num_sats_std
0,0,1.0,0.457513,0.833354,0.179008,893.0,513.0,707.333333,128.776140,216.0,24.0,83.555556,47.250549
1,1,1.0,0.457513,0.875205,0.161843,893.0,513.0,717.961538,127.177822,286.0,24.0,96.615385,64.048155
2,2,1.0,0.284882,0.853851,0.185964,893.0,513.0,708.100000,125.052913,286.0,16.0,92.266667,61.832328
3,3,1.0,0.284882,0.871325,0.181294,943.0,513.0,734.941176,130.240324,286.0,16.0,95.647059,64.844748
4,4,1.0,0.284882,0.877865,0.178169,943.0,513.0,722.361111,131.214362,286.0,16.0,100.555556,68.891818
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1.0,0.199353,0.818133,0.225847,997.0,503.0,700.865546,154.552877,216.0,10.0,75.915966,45.246328
97,97,1.0,0.199353,0.819189,0.225193,997.0,503.0,702.475000,154.908700,216.0,10.0,75.750000,45.092482
98,98,1.0,0.199353,0.817877,0.225785,997.0,503.0,699.303279,155.591268,216.0,10.0,76.393443,45.766847
99,99,1.0,0.199353,0.817344,0.225004,997.0,503.0,700.508065,155.378033,216.0,10.0,75.935484,45.560777


In [48]:
export_dataframes_to_csv(all_hall_of_fame_gene_stats_dfs, "/content/vietnam-satellite-simulation/hof_logbook", "hof_logbook")

✅ Exported hof_logbook_0.csv
✅ Exported hof_logbook_50.csv
✅ Exported hof_logbook_60.csv
✅ Exported hof_logbook_70.csv
✅ Exported hof_logbook_80.csv
✅ Exported hof_logbook_90.csv
✅ Exported hof_logbook_95.csv
✅ Exported hof_logbook_99.csv
✅ Exported hof_logbook_100.csv
✅ Exported hof_logbook_2016.csv


In [ ]:
# @title
def plot_average_hof_fitness_evolution(all_hall_of_fame_gene_stats_dfs, output_folder):
    """
    Generates line plots comparing the evolution of the average and standard deviation of fitness
    (Coverage, Altitude, Số vệ tinh) in the Hall of Fame across all loaded checkpoints.

    Parameters
    ----------
    all_hall_of_fame_gene_stats_dfs : dict
        A dictionary where keys are checkpoint integers and values are the
        corresponding DataFrames containing the gene and fitness statistics
        for the Hall of Fame history of a specific checkpoint (as created by
        the cell adb14b56).
    output_folder : str
        The path to the folder where the plots should be saved.
    """
    if not all_hall_of_fame_gene_stats_dfs:
        print("Không có dữ liệu thống kê Hall of Fame để vẽ biểu đồ trung bình fitness.")
        return

    fitness_objectives = [
        "Fitness (Coverage)",
        "Fitness (Altitude)",
        "Fitness (Số vệ tinh)"
    ]

    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    sorted_keys = sorted(all_hall_of_fame_gene_stats_dfs.keys())
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(sorted_keys):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(sorted_keys)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # Define markers and linestyles for plot differentiation
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']

    # MỚI: Iterate through each objective and each statistic type (mean, std)
    for fitness_name in fitness_objectives:
        for stat_type in ['mean', 'std']:
            plt.figure(figsize=(12, 7))
            title = f"Evolution of {stat_type.title()} Hall of Fame {fitness_name.replace('Fitness', '').strip()} across Checkpoints"
            ylabel = f"{fitness_name} ({stat_type.title()})"

            plotted_any = False
            # Iterate through the dictionary items (checkpoint_key, DataFrame)
            # MỚI: Dùng enumerate trên list đã sắp xếp
            for i, checkpoint_key in enumerate(sorted_keys):
                hall_of_fame_gene_stats = all_hall_of_fame_gene_stats_dfs[checkpoint_key]

                # MỚI: Quyết định label
                if use_custom_labels:
                    checkpoint_name = CUSTOM_LABELS[i]
                else:
                    checkpoint_name = f"Test {checkpoint_key}" # Fallback
                # print(f"Plotting for {checkpoint_name}") # Debugging line

                avg_col_name = f"{fitness_name}_{stat_type}"

                if avg_col_name in hall_of_fame_gene_stats.columns:
                    # MỚI: Use dynamic markers and linestyles
                    marker = markers[i % len(markers)]
                    linestyle = linestyles[i % len(linestyles)]

                    plt.plot(hall_of_fame_gene_stats['Generation'], hall_of_fame_gene_stats[avg_col_name],
                             label=checkpoint_name,
                             marker=marker,
                             linestyle=linestyle,
                             markersize=5)
                    plotted_any = True
                else:
                    print(f"Cột '{avg_col_name}' không tìm thấy trong thống kê Hall of Fame từ {checkpoint_name}. Bỏ qua plotting.")

            if plotted_any:
                plt.gca().spines[['top', 'right',]].set_visible(False) # MỚI
                plt.xlabel("Generation")
                plt.ylabel(ylabel)
                plt.title(title)
                plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left') # MỚI
                plt.grid(True)

                # Define the filename
                filename = f"comparison_{stat_type}_hof_{fitness_name.replace(' ', '_').replace('(', '').replace(')', '')}_plot.png"
                filepath = os.path.join(output_folder, filename)

                # Save the figure
                plt.savefig(filepath, bbox_inches='tight') # MỚI
                print(f"✅ Saved plot to {filepath}")
                plt.close() # Close the figure to free up memory
            else:
                print(f"Không có dữ liệu để vẽ biểu đồ {stat_type} fitness cho {fitness_name}. Bỏ qua tạo plot.")

plot_average_hof_fitness_evolution(all_hall_of_fame_gene_stats_dfs, OUTPUT_FOLDER)

✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_mean_hof_Fitness_Coverage_plot.png
✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_std_hof_Fitness_Coverage_plot.png
✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_mean_hof_Fitness_Altitude_plot.png
✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_std_hof_Fitness_Altitude_plot.png
✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_mean_hof_Fitness_Số_vệ_tinh_plot.png
✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_std_hof_Fitness_Số_vệ_tinh_plot.png


Biểu đồ này giúp đánh giá mức độ cải tiến của thuật toán theo thời gian — nếu càng về sau số cá thể mới giảm dần, điều đó cho thấy thuật toán đã gần đạt tới sự ổn định.

## **9. Phân tích và Lựa chọn Kết quả Tối ưu (Hall of Fame)**

Sau khi quá trình tiến hoá hoàn tất, tất cả các lời giải ưu tú nhất được lưu trong Hall of Fame (HOF). Phần này tập trung vào việc phân tích mặt trận Pareto, đánh giá sự đánh đổi (trade-off) giữa các mục tiêu, và lựa chọn lời giải tối ưu cuối cùng để xuất dữ liệu TLE hoặc sử dụng cho các mô phỏng quỹ đạo.

### **9.1. Mặt trận Pareto - Sự đánh đổi giữa các Mục tiêu**

Trong tối ưu đa mục tiêu, mặt trận Pareto (Pareto Front) biểu diễn tập hợp các lời giải mà không có cá thể nào trội hơn toàn bộ các cá thể khác trên tất cả các mục tiêu.

Để trực quan hoá mặt trận Pareto, ta chuyển danh sách Hall of Fame thành DataFrame.

In [30]:
# @title
import pandas as pd

# Create a dictionary to hold the final hall_of_fame_df DataFrames for all checkpoints
# The key will be the checkpoint key (integer)
all_final_hall_of_fame_dfs = {}

# Iterate through the loaded checkpoints data
if 'loaded_checkpoints_data' not in locals() or not loaded_checkpoints_data:
    print("Không có dữ liệu checkpoint nào được tải. Vui lòng chạy cell tải checkpoint trước.")
else:
    # Sort the checkpoint keys (integers) to process them in order
    sorted_checkpoint_keys = sorted(loaded_checkpoints_data.keys())

    for checkpoint_key in sorted_checkpoint_keys:
        if checkpoint_key == 2016:
            continue
        data = loaded_checkpoints_data[checkpoint_key]
        halloffame = data["halloffame"]
        # Assuming logbook is also available in the loaded data for 'new_hof' if needed,
        # though 'new_hof' is not used in the final HOF DataFrame.
        # logbook = data.get("logbook") # Get logbook if available

        # Create a list to hold the data for the DataFrame for the current checkpoint
        hall_of_fame_data = []

        # Iterate through the individuals in the hall of fame
        for i, ind in enumerate(halloffame):
            # Extract parameters
            altitude = ind[0]
            inclination = ind[1]
            num_planes = int(ind[2])
            sats_per_plane = int(ind[3])
            phasing = ind[4]

            # Extract fitness values
            # Assuming the fitness values are in the order (coverage, altitude, num_sats)
            coverage = ind.fitness.values[0]
            altitude_fitness = ind.fitness.values[1]
            costs = ind.fitness.values[2] # Use a different name to avoid conflict

            # Append the data as a dictionary to the list
            hall_of_fame_data.append({
                "Checkpoint_Key": checkpoint_key, # Add checkpoint key as a column
                "Checkpoint_File": os.path.basename(CHECKPOINT_FILE[sorted_checkpoint_keys.index(checkpoint_key)]), # Get original file name
                "Lời giải": i + 1,
                "Altitude (km)": altitude,
                "Inclination (deg)": inclination,
                "Số mặt phẳng quỹ đạo": num_planes,
                "Số vệ tinh mỗi mặt phẳng": sats_per_plane,
                "Phasing": phasing,
                "Fitness (Coverage)": coverage,
                "Fitness (Altitude)": altitude_fitness,
                "Fitness (Số vệ tinh)": int(costs)
            })

        # Create the pandas DataFrame for the current checkpoint's final hall of fame
        hall_of_fame_df = pd.DataFrame(hall_of_fame_data)

        # Store the DataFrame in the dictionary with the checkpoint key
        all_final_hall_of_fame_dfs[checkpoint_key] = hall_of_fame_df

# Display the dictionary of DataFrames (optional)
print("Dictionary of final_hall_of_fame_df DataFrames:")
for key, df in all_final_hall_of_fame_dfs.items():
    print(f"\nDataFrame from checkpoint key {key}:")
    display(df)

# The subsequent cells that use all_final_hall_of_fame_dfs will need to be updated
# to iterate through the dictionary instead of a list.

Dictionary of final_hall_of_fame_df DataFrames:

DataFrame from checkpoint key 0:


,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,0,checkpoint_final_constraint_0.pkl,1,500,20,12,15,2,1.000000,500.0,180
1,0,checkpoint_final_constraint_0.pkl,2,563,19,12,13,4,1.000000,563.0,156
2,0,checkpoint_final_constraint_0.pkl,3,575,19,10,15,1,1.000000,575.0,150
3,0,checkpoint_final_constraint_0.pkl,4,575,19,10,15,0,1.000000,575.0,150
4,0,checkpoint_final_constraint_0.pkl,5,625,19,5,24,4,1.000000,625.0,120
...,...,...,...,...,...,...,...,...,...,...,...
955,0,checkpoint_final_constraint_0.pkl,956,563,21,1,1,0,0.003883,563.0,1
956,0,checkpoint_final_constraint_0.pkl,957,550,20,1,1,0,0.003606,550.0,1
957,0,checkpoint_final_constraint_0.pkl,958,538,21,1,1,0,0.003514,538.0,1
958,0,checkpoint_final_constraint_0.pkl,959,529,21,1,1,0,0.003236,529.0,1



DataFrame from checkpoint key 50:


,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,50,checkpoint_final_constraint_50.pkl,1,501,19,9,22,2,1.000000,501.0,198
1,50,checkpoint_final_constraint_50.pkl,2,511,20,12,15,5,1.000000,511.0,180
2,50,checkpoint_final_constraint_50.pkl,3,511,19,12,15,5,1.000000,511.0,180
3,50,checkpoint_final_constraint_50.pkl,4,533,20,7,23,3,1.000000,533.0,161
4,50,checkpoint_final_constraint_50.pkl,5,599,20,7,22,1,1.000000,599.0,154
...,...,...,...,...,...,...,...,...,...,...,...
767,50,checkpoint_final_constraint_50.pkl,768,805,19,2,1,1,0.048636,805.0,2
768,50,checkpoint_final_constraint_50.pkl,769,650,18,3,1,2,0.045123,650.0,3
769,50,checkpoint_final_constraint_50.pkl,770,769,15,2,1,1,0.042534,769.0,2
770,50,checkpoint_final_constraint_50.pkl,771,802,21,1,1,1,0.009986,802.0,1



DataFrame from checkpoint key 60:


...

,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,95,checkpoint_final_constraint_95.pkl,1,500,19,12,17,3,1.000000,500.0,204
1,95,checkpoint_final_constraint_95.pkl,2,531,20,12,14,6,1.000000,531.0,168
2,95,checkpoint_final_constraint_95.pkl,3,548,19,11,15,3,1.000000,548.0,165
3,95,checkpoint_final_constraint_95.pkl,4,548,20,11,15,3,1.000000,548.0,165
4,95,checkpoint_final_constraint_95.pkl,5,583,22,7,21,2,1.000000,583.0,147
...,...,...,...,...,...,...,...,...,...,...,...
471,95,checkpoint_final_constraint_95.pkl,472,513,17,5,1,2,0.058992,513.0,5
472,95,checkpoint_final_constraint_95.pkl,473,667,18,4,1,3,0.052982,667.0,4
473,95,checkpoint_final_constraint_95.pkl,474,623,18,4,1,3,0.048729,623.0,4
474,95,checkpoint_final_constraint_95.pkl,475,802,21,1,1,1,0.009986,802.0,1



DataFrame from checkpoint key 99:


,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,99,checkpoint_final_constraint_99.pkl,1,501,19,11,17,1,1.000000,501.0,187
1,99,checkpoint_final_constraint_99.pkl,2,527,21,10,18,1,1.000000,527.0,180
2,99,checkpoint_final_constraint_99.pkl,3,550,21,7,23,3,1.000000,550.0,161
3,99,checkpoint_final_constraint_99.pkl,4,582,23,9,16,1,1.000000,582.0,144
4,99,checkpoint_final_constraint_99.pkl,5,602,21,7,18,2,1.000000,602.0,126
...,...,...,...,...,...,...,...,...,...,...,...
364,99,checkpoint_final_constraint_99.pkl,365,796,16,3,1,1,0.067684,796.0,3
365,99,checkpoint_final_constraint_99.pkl,366,502,20,1,23,0,0.064448,502.0,23
366,99,checkpoint_final_constraint_99.pkl,367,650,19,4,1,3,0.052427,650.0,4
367,99,checkpoint_final_constraint_99.pkl,368,602,21,2,1,1,0.030791,602.0,2



DataFrame from checkpoint key 100:


,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,100,checkpoint_final_constraint_100.pkl,1,511,19,9,20,3,1.000000,511.0,180
1,100,checkpoint_final_constraint_100.pkl,2,607,19,6,22,1,1.000000,607.0,132
2,100,checkpoint_final_constraint_100.pkl,3,745,19,6,21,1,1.000000,745.0,126
3,100,checkpoint_final_constraint_100.pkl,4,772,19,12,7,10,1.000000,772.0,84
4,100,checkpoint_final_constraint_100.pkl,5,772,20,12,7,1,1.000000,772.0,84
...,...,...,...,...,...,...,...,...,...,...,...
230,100,checkpoint_final_constraint_100.pkl,231,772,19,2,1,1,0.045677,772.0,2
231,100,checkpoint_final_constraint_100.pkl,232,665,22,1,4,1,0.020620,665.0,4
232,100,checkpoint_final_constraint_100.pkl,233,502,19,1,7,0,0.018031,502.0,7
233,100,checkpoint_final_constraint_100.pkl,234,607,19,1,5,0,0.017753,607.0,5


In [41]:
[]

,Checkpoint_Key,Checkpoint_File,Generation,Individual_Index,New Individuals in Hall of Fame,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,0,checkpoint_final_constraint_0.pkl,0,1,33,691,22,9,15,2,1.000000,691.0,135.0
1,0,checkpoint_final_constraint_0.pkl,0,2,33,882,24,7,19,4,1.000000,882.0,133.0
2,0,checkpoint_final_constraint_0.pkl,0,3,33,852,23,4,22,2,0.999075,852.0,88.0
3,0,checkpoint_final_constraint_0.pkl,0,4,33,552,25,12,18,2,0.998890,552.0,216.0
4,0,checkpoint_final_constraint_0.pkl,0,5,33,805,16,7,13,5,0.992880,805.0,91.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60339,0,checkpoint_final_constraint_0.pkl,100,956,9,563,21,1,1,0,0.003883,563.0,1.0
60340,0,checkpoint_final_constraint_0.pkl,100,957,9,550,20,1,1,0,0.003606,550.0,1.0
60341,0,checkpoint_final_constraint_0.pkl,100,958,9,538,21,1,1,0,0.003514,538.0,1.0
60342,0,checkpoint_final_constraint_0.pkl,100,959,9,529,21,1,1,0,0.003236,529.0,1.0


In [31]:
# @title
import pandas as pd
from deap import base, tools, creator # Import necessary DEAP modules

# Ensure DEAP creator is set up (assuming it was done in a previous cell)
if not hasattr(creator, "FitnessMulti"):
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMulti)

# Check if loaded_checkpoints_data exists and contains key 2016
roullete_key = 2016
if 'loaded_checkpoints_data' in locals() and roullete_key in loaded_checkpoints_data:
    print(f"Processing halloffame history for checkpoint key: {roullete_key}")
    roullete_data = loaded_checkpoints_data[roullete_key]
    halloffame_history = roullete_data.get('halloffame_history')

    if halloffame_history and isinstance(halloffame_history, list):
        all_individuals_from_hof_history = []
        for gen_data in halloffame_history:
            halloffame_list = gen_data['halloffame']
            fitnesses_list = gen_data['fitnesses']

            # Convert individuals and fitnesses to DEAP individuals
            for i in range(len(halloffame_list)):
                ind = creator.Individual(halloffame_list[i])
                ind.fitness.values = fitnesses_list[i]
                ind.fitness.cvalues = (0,) # Placeholder, assuming HOF individuals satisfy constraints
                # Add checkpoint file information
                ind.checkpoint_file = f"checkpoint_final_constraint_{roullete_key}" # Use the key as the file identifier
                ind.original_index = i + 1 # Use index within the generation's HOF as original index
                all_individuals_from_hof_history.append(ind)

        if all_individuals_from_hof_history:
            print(f"Found {len(all_individuals_from_hof_history)} individuals in halloffame history. Applying non-dominated sort.")
            # Apply non-dominated sorting to find the Pareto front
            # Use the fastConstrainedNondominatedSort function defined earlier
            # Get the first front (Pareto front)
            all_fronts = fastConstrainedNondominatedSort(all_individuals_from_hof_history, len(all_individuals_from_hof_history), first_front_only=False)
            roullete_pareto_front_individuals = all_fronts[0] # The first front is the desired non-dominated set

            # Convert the resulting Pareto front individuals back to a DataFrame
            roullete_hall_of_fame_data = []
            for ind in roullete_pareto_front_individuals:
                roullete_hall_of_fame_data.append({
                    "Checkpoint_Key": roullete_key, # Use the integer key
                    "Checkpoint_File": ind.checkpoint_file,
                    "Lời giải": ind.original_index, # Use the original index from history
                    "Altitude (km)": ind[0],
                    "Inclination (deg)": ind[1],
                    "Số mặt phẳng quỹ đạo": int(ind[2]),
                    "Số vệ tinh mỗi mặt phẳng": int(ind[3]),
                    "Phasing": ind[4],
                    "Fitness (Coverage)": ind.fitness.values[0],
                    "Fitness (Altitude)": ind.fitness.values[1],
                    "Fitness (Số vệ tinh)": int(ind.fitness.values[2]),
                })

            roullete_hall_of_fame_df = pd.DataFrame(roullete_hall_of_fame_data)

            # Drop duplicate rows based on all columns except 'Lời giải' and the index
            columns_to_consider_for_duplicates = [col for col in roullete_hall_of_fame_df.columns if col not in ['Lời giải']]
            roullete_hall_of_fame_df = roullete_hall_of_fame_df.drop_duplicates(subset=columns_to_consider_for_duplicates, keep='first')
            roullete_hall_of_fame_df = roullete_hall_of_fame_df.reset_index(drop=True) # Reset index after dropping duplicates


            # Add this DataFrame to the all_final_hall_of_fame_dfs dictionary
            # Ensure all_final_hall_of_fame_dfs exists, initialize if not
            if 'all_final_hall_of_fame_dfs' not in locals():
                all_final_hall_of_fame_dfs = {}

            all_final_hall_of_fame_dfs[roullete_key] = roullete_hall_of_fame_df

            print(f"✅ Đã thêm Hall of Fame (Pareto Front từ lịch sử) của checkpoint {roullete_key} vào all_final_hall_of_fame_dfs (đã loại bỏ trùng lặp).")
            display(all_final_hall_of_fame_dfs[roullete_key]) # Display head of the new DataFrame

        else:
            print(f"⚠️ Halloffame history for checkpoint key {roullete_key} is empty or not in list format. Cannot extract Pareto front.")

else:
    print("⚠️ 'loaded_checkpoints_data' not found or does not contain key 2016. Cannot process halloffame history.")

# The rest of the analysis cells will now automatically include data for key 2016
# if they iterate through all_final_hall_of_fame_dfs.

Processing halloffame history for checkpoint key: 2016
Found 8126 individuals in halloffame history. Applying non-dominated sort.
✅ Đã thêm Hall of Fame (Pareto Front từ lịch sử) của checkpoint 2016 vào all_final_hall_of_fame_dfs (đã loại bỏ trùng lặp).


,Checkpoint_Key,Checkpoint_File,Lời giải,Altitude (km),Inclination (deg),Số mặt phẳng quỹ đạo,Số vệ tinh mỗi mặt phẳng,Phasing,Fitness (Coverage),Fitness (Altitude),Fitness (Số vệ tinh)
0,2016,checkpoint_final_constraint_2016,7,772.0,17.0,3,22,2.0,0.942672,772.0,66
1,2016,checkpoint_final_constraint_2016,9,857.0,21.0,6,9,2.0,0.925751,857.0,54
2,2016,checkpoint_final_constraint_2016,11,513.0,23.0,4,23,3.0,0.833934,513.0,92
3,2016,checkpoint_final_constraint_2016,12,874.0,22.0,14,3,13.0,0.804253,874.0,42
4,2016,checkpoint_final_constraint_2016,13,685.0,18.0,3,17,2.0,0.791586,685.0,51
...,...,...,...,...,...,...,...,...,...,...,...
116,2016,checkpoint_final_constraint_2016,52,894.0,18.0,4,14,2.0,0.944799,894.0,56
117,2016,checkpoint_final_constraint_2016,105,509.0,23.0,10,5,4.0,0.493111,509.0,50
118,2016,checkpoint_final_constraint_2016,51,895.0,24.0,12,5,3.0,0.952843,895.0,60
119,2016,checkpoint_final_constraint_2016,100,653.0,20.0,9,4,7.0,0.616736,653.0,36


In [32]:
# @title
# Print the number of individuals with 100% coverage in the final Hall of Fame for each checkpoint
if 'all_final_hall_of_fame_dfs' in locals() and all_final_hall_of_fame_dfs:
    print("Số lượng cá thể có độ phủ 100% trong Hall of Fame cuối cùng:")
    # Sort the checkpoint keys (integers) for consistent output order
    sorted_checkpoint_keys = sorted(all_final_hall_of_fame_dfs.keys())

    for checkpoint_key in sorted_checkpoint_keys:
        hall_of_fame_df = all_final_hall_of_fame_dfs[checkpoint_key]
        # Filter for individuals with 100% coverage (or close to 100% due to floating point precision)
        coverage_100_percent_individuals = hall_of_fame_df[hall_of_fame_df['Fitness (Coverage)'] >= 1.0]
        count_100_percent = len(coverage_100_percent_individuals)
        print(f"Test {checkpoint_key}: {count_100_percent} cá thể")
else:
    print("Không có dữ liệu Hall of Fame cuối cùng được tìm thấy. Vui lòng chạy cell tạo final Hall of Fame DataFrame trước.")

Số lượng cá thể có độ phủ 100% trong Hall of Fame cuối cùng:
Test 0: 11 cá thể
Test 50: 24 cá thể
Test 60: 14 cá thể
Test 70: 23 cá thể
Test 80: 16 cá thể
Test 90: 15 cá thể
Test 95: 17 cá thể
Test 99: 12 cá thể
Test 100: 7 cá thể
Test 2016: 11 cá thể


### **9.2. Trực quan hóa Sự đánh đổi (Trade-off Visualization)**

Phân tích chi tiết hơn cho phép hiểu rõ mức độ ảnh hưởng giữa các mục tiêu. Phần này trực quan hóa các tương quan cặp mục tiêu, qua đó hỗ trợ ra quyết định lựa chọn lời giải tối ưu cân bằng.

##101. ĐÁNH GIÁ PHÂN BỐ

In [69]:
# @title
import collections
import pandas as pd
from tqdm import tqdm # Assuming tqdm is imported from earlier cells

def calculate_simpsons_diversity_index_d(population):
    """
    Calculates Simpson's Diversity Index (D) for a given population.

    Simpson's Index (D) measures the probability that two individuals
    randomly selected from a sample will belong to the same species (type).
    A higher D value indicates lower diversity.

    Formula: D = sum(n_i * (n_i - 1)) / (N * (N - 1))
    where:
        n_i = number of individuals of type i
        N = total number of individuals in the sample

    In this context, "type" refers to a unique individual configuration
    (gene values).

    Parameters
    ----------
    population : list
        A list of individuals (each individual is a list of gene values).

    Returns
    -------
    float
        Simpson's Diversity Index (D) for the population.
        Returns 0.0 if the population size is less than 2.
    """
    N = len(population)
    if N < 2:
        return 0.0

    # Count the occurrences of each unique individual type (gene configuration)
    # Convert individual list to tuple to make it hashable for Counter
    individual_counts = collections.Counter(tuple(ind) for ind in population)

    # Calculate sum of n_i * (n_i - 1)
    sum_ni_ni_minus_1 = sum(count * (count - 1) for count in individual_counts.values())

    # Calculate Simpson's Index (D)
    simpsons_d = sum_ni_ni_minus_1 / (N * (N - 1))

    return simpsons_d

# Calculate and store Simpson's Index for each generation in pop_history for each checkpoint
simpsons_diversity_dataframes = {} # This will continue to store individual DFs for plotting
combined_simpsons_diversity_df = None # This will be the single combined DataFrame requested by the user
all_simpsons_dfs_for_merge = [] # This will be used to build the combined DF

if 'loaded_checkpoints_data' in locals() and loaded_checkpoints_data:
    print("Calculating Simpson's Diversity Index for each generation across checkpoints...")

    # Create a mapping from checkpoint key (integer) to its CUSTOM_LABEL
    labels_to_use = {}
    if 'CUSTOM_LABELS' in globals() and 'CHECKPOINT_FILES' in globals() and \
       len(CUSTOM_LABELS) == len(CHECKPOINT_FILES):
        labels_to_use = {key: label for key, label in zip(CHECKPOINT_FILES, CUSTOM_LABELS)}
    else:
        print("Warning: CUSTOM_LABELS or CHECKPOINT_FILES not properly defined or lengths mismatch. Using default keys for column names.")
        # Fallback: dynamically get keys from loaded_checkpoints_data
        for key in sorted(loaded_checkpoints_data.keys()):
            labels_to_use[key] = f"Test {key}"

    sorted_checkpoint_keys = sorted(loaded_checkpoints_data.keys()) # Ensure processing in a consistent order

    for checkpoint_key in sorted_checkpoint_keys:
        if checkpoint_key in loaded_checkpoints_data:
            data = loaded_checkpoints_data[checkpoint_key]
            pop_history = data['pop_history']
            simpsons_diversity_history = []

            checkpoint_label = labels_to_use.get(checkpoint_key, f"Test {checkpoint_key}")

            for gen_data in tqdm(pop_history, desc=f"Processing Generations for {checkpoint_label}"):
                generation = gen_data['gen']
                population_data = gen_data['population']
                simpsons_d = calculate_simpsons_diversity_index_d(population_data)
                simpsons_diversity_history.append({'generation': generation, 'Simpsons_D': simpsons_d})

            temp_df = pd.DataFrame(simpsons_diversity_history)
            if not temp_df.empty:
                # Store individual DF for plotting functions that might expect a dict of DFs
                simpsons_diversity_dataframes[checkpoint_key] = temp_df.copy()

                # Prepare for the combined DataFrame: rename 'Simpsons_D' to its custom label
                temp_df_for_merge = temp_df.rename(columns={'Simpsons_D': checkpoint_label})
                all_simpsons_dfs_for_merge.append(temp_df_for_merge)
        else:
            print(f"Warning: Checkpoint key {checkpoint_key} not found in loaded_checkpoints_data. Skipping.")

    # Display the resulting individual DataFrames (optional, for verification by other cells)
    print("\nSimpson's Diversity Index per Generation for each checkpoint (individual DataFrames):")
    # for key, df in simpsons_diversity_dataframes.items():
    #     print(f"\nCheckpoint key {key}:")
    #     display(df)

    # --- Create the single combined DataFrame as requested by the user ---
    if all_simpsons_dfs_for_merge:
        combined_simpsons_diversity_df = all_simpsons_dfs_for_merge[0]
        for i in range(1, len(all_simpsons_dfs_for_merge)):
            combined_simpsons_diversity_df = pd.merge(
                combined_simpsons_diversity_df,
                all_simpsons_dfs_for_merge[i],
                on='generation',
                how='outer'
            )
        combined_simpsons_diversity_df.sort_values(by='generation', inplace=True)
        combined_simpsons_diversity_df.fillna(0, inplace=True) # Fill NaNs if some generations are missing

        print("\nCombined Simpson's Diversity Index per Generation across all checkpoints:")
        display(combined_simpsons_diversity_df)

    else:
        print("No data found to combine for Simpson's Diversity Index into a single DataFrame.")

else:
    print("Error: 'loaded_checkpoints_data' not found or is empty. Please ensure the checkpoints have been loaded.")

Calculating Simpson's Diversity Index for each generation across checkpoints...


Processing Generations for MOGA-WS: 100%|██████████| 101/101 [00:00<00:00, 7752.16it/s]


Simpson's Diversity Index per Generation for each checkpoint (individual DataFrames):



Combined Simpson's Diversity Index per Generation across all checkpoints:


,generation,NSGA-NC,NSGA-C50,NSGA-C60,NSGA-C70,NSGA-C80,NSGA-C90,NSGA-C95,NSGA-C99,NSGA-C100,MOGA-WS
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.029388
1,1,0.000000,0.000816,0.000000,0.000000,0.000816,0.000816,0.000000,0.000000,0.000000,0.017143
2,2,0.000000,0.001633,0.001633,0.000816,0.000816,0.003265,0.003265,0.002449,0.002449,0.015510
3,3,0.000816,0.001633,0.001633,0.000000,0.003265,0.003265,0.004898,0.003265,0.004898,0.015510
4,4,0.000816,0.000000,0.000000,0.000000,0.006531,0.004082,0.005714,0.004082,0.008163,0.015510
...,...,...,...,...,...,...,...,...,...,...,...
96,96,0.001633,0.001633,0.000816,0.001633,0.000816,0.001633,0.004082,0.004082,0.706122,0.013878
97,97,0.002449,0.001633,0.000000,0.000816,0.000816,0.001633,0.001633,0.006531,0.706122,0.021224
98,98,0.001633,0.000816,0.000000,0.000000,0.000816,0.003265,0.001633,0.007347,0.706122,0.017959
99,99,0.002449,0.000816,0.000000,0.000816,0.000816,0.003265,0.002449,0.007347,0.706122,0.026939


In [70]:
export_dataframes_to_csv(combined_simpsons_diversity_df, "/content/vietnam-satellite-simulation/diversity", "diversity", False)

✅ Exported diversity.csv


In [ ]:
# @title
import matplotlib.pyplot as plt
import os

def plot_simpsons_diversity_per_checkpoint(simpsons_diversity_dataframes, output_folder):
    """
    Generates scatter plots of Simpson's Diversity Index per Generation
    for each checkpoint and saves them to the output folder.
    (Đã cập nhật để dùng CUSTOM_LABELS cho tiêu đề)
    """
    if not simpsons_diversity_dataframes:
        print("Không có dữ liệu chỉ số đa dạng Simpson để vẽ biểu đồ.")
        return

    os.makedirs(output_folder, exist_ok=True)

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(simpsons_diversity_dataframes):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(simpsons_diversity_dataframes)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # MỚI: Dùng enumerate để lấy index (i)
    for i, (checkpoint_key, simpsons_diversity_df) in enumerate(simpsons_diversity_dataframes.items()):
        if 'Generation' in simpsons_diversity_df.columns and 'Simpsons_D' in simpsons_diversity_df.columns:
            plt.figure(figsize=(10, 6))
            plt.scatter(simpsons_diversity_df['Generation'], simpsons_diversity_df['Simpsons_D'], s=32, alpha=.8)
            plt.gca().spines[['top', 'right',]].set_visible(False)

            # MỚI: Quyết định tên cho tiêu đề
            if use_custom_labels:
                checkpoint_name = CUSTOM_LABELS[i]
            else:
                checkpoint_name = f"Test {checkpoint_key}" # Fallback

            plt.xlabel("Generation")
            plt.ylabel("Simpson's Diversity Index (D)")
            # MỚI: Sử dụng checkpoint_name (từ CUSTOM_LABELS) trong tiêu đề
            plt.title(f"Simpson's Diversity Index per Generation ({checkpoint_name})")
            plt.grid(True)

            # Giữ nguyên tên file dựa trên key để đảm bảo tính duy nhất và đơn giản
            filename = f"test_{checkpoint_key}_simpsons_diversity_plot.png"
            filepath = os.path.join(output_folder, filename)

            plt.savefig(filepath)
            print(f"✅ Saved plot to {filepath}")
            plt.close()
        else:
            # MỚI: Cập nhật thông báo lỗi nếu có thể
            label_for_error = CUSTOM_LABELS[i] if use_custom_labels else f"Test {checkpoint_key}"
            print(f"Skipping plot for {label_for_error}: required columns 'Generation' or 'Simpsons_D' not found.")

def plot_simpsons_diversity_comparison(simpsons_diversity_dataframes, output_folder):
    """
    Generates a single plot comparing Simpson's Diversity Index per Generation
    across all loaded checkpoints and saves it to the output folder.
    (Đã cập nhật để cải thiện visual và dùng CUSTOM_LABELS)
    """
    if not simpsons_diversity_dataframes:
        print("Không có dữ liệu chỉ số đa dạng Simpson để vẽ biểu đồ so sánh.")
        return

    # MỚI: Tăng kích thước, thêm markers và linestyles
    plt.figure(figsize=(14, 7))
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']
    plotted_any = False

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(simpsons_diversity_dataframes):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(simpsons_diversity_dataframes)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # MỚI: Dùng enumerate để lấy index (i)
    for i, (checkpoint_key, simpsons_diversity_df) in enumerate(simpsons_diversity_dataframes.items()):
        if 'Generation' in simpsons_diversity_df.columns and 'Simpsons_D' in simpsons_diversity_df.columns:

            # MỚI: Quyết định label
            if use_custom_labels:
                label_name = CUSTOM_LABELS[i]
            else:
                label_name = f"Test {checkpoint_key}" # Fallback

            # MỚI: Chọn marker và linestyle
            marker = markers[i % len(markers)]
            linestyle = linestyles[i % len(linestyles)]

            # MỚI: Cập nhật lệnh plot
            plt.plot(simpsons_diversity_df['Generation'], simpsons_diversity_df['Simpsons_D'],
                     label=label_name,
                     marker=marker,
                     linestyle=linestyle,
                     markersize=5)
            plotted_any = True
        else:
            label_for_error = CUSTOM_LABELS[i] if use_custom_labels else f"Test {checkpoint_key}"
            print(f"Skipping plot for {label_for_error}: required columns 'Generation' or 'Simpsons_D' not found.")

    if plotted_any:
        plt.gca().spines[['top', 'right',]].set_visible(False)
        plt.xlabel("Generation")
        plt.ylabel("Simpson's Diversity Index (D)")
        plt.title("Comparison of Simpson's Diversity Index per Generation across Checkpoints")

        # MỚI: Cập nhật legend
        plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.grid(True)

        os.makedirs(output_folder, exist_ok=True)

        filename = "comparison_simpsons_diversity_plot.png"
        filepath = os.path.join(output_folder, filename)

        # MỚI: Cập nhật savefig
        plt.savefig(filepath, bbox_inches='tight')
        print(f"✅ Saved comparison plot to {filepath}")
        plt.close()
    else:
        print("Không có dữ liệu để vẽ biểu đồ so sánh chỉ số đa dạng Simpson.")


# Example usage (assuming simpsons_diversity_dataframes, OUTPUT_FOLDER,
# and CUSTOM_LABELS are defined in previous cells):
#
# plot_simpsons_diversity_per_checkpoint(simpsons_diversity_dataframes, OUTPUT_FOLDER)
plot_simpsons_diversity_comparison(simpsons_diversity_dataframes, OUTPUT_FOLDER)

✅ Saved comparison plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_simpsons_diversity_plot.png


In [65]:
# @title
def count_100_percent_coverage_individuals(pop_history):
    """
    Counts the number of individuals with 100% coverage in each generation
    from the pop_history.

    Parameters
    ----------
    pop_history : list
        A list of dictionaries, where each dictionary represents a generation's
        data, including 'gen' and 'fitnesses'.

    Returns
    -------
    list of dict
        A list of dictionaries, each containing the 'Generation' and the
        '100% Coverage Count' for that generation.
    """
    coverage_counts = []
    if pop_history:
        for gen_data in pop_history:
            generation = gen_data['gen']
            fitnesses = gen_data['fitnesses'] # This is a list of tuples (coverage, altitude, num_sats)

            count_100_percent = sum(1 for fitness in fitnesses if fitness[0] >= 1.0) # Count if coverage is 1.0 or more

            coverage_counts.append({'generation': generation, '100% Coverage Count': count_100_percent})
    return coverage_counts

# Calculate and store 100% coverage counts for each generation in pop_history for each checkpoint
# This will now create a single pandas DataFrame as requested by the user.
combined_100_coverage_counts_df = None
all_coverage_dfs_for_merge = []

if 'loaded_checkpoints_data' in locals() and loaded_checkpoints_data:
    print("Calculating 100% Coverage Counts for each generation across checkpoints and combining into a single DataFrame...")

    # Ensure CUSTOM_LABELS and CHECKPOINT_FILES are available and correctly mapped
    if 'CUSTOM_LABELS' not in globals() or 'CHECKPOINT_FILES' not in globals() or \
       len(CUSTOM_LABELS) != len(CHECKPOINT_FILES):
        print("Warning: CUSTOM_LABELS or CHECKPOINT_FILES not properly defined or lengths mismatch. Using default keys for columns.")
        # Fallback to using checkpoint_key as column name if CUSTOM_LABELS/CHECKPOINT_FILES are not aligned
        sorted_checkpoint_keys = sorted(loaded_checkpoints_data.keys())
        labels_to_use = {key: f"Test {key}" for key in sorted_checkpoint_keys}
    else:
        # Create a mapping from checkpoint key (integer) to its CUSTOM_LABEL
        labels_to_use = {key: label for key, label in zip(CHECKPOINT_FILES, CUSTOM_LABELS)}
        sorted_checkpoint_keys = CHECKPOINT_FILES # Use the provided order from CHECKPOINT_FILES

    for checkpoint_key in sorted_checkpoint_keys:
        if checkpoint_key in loaded_checkpoints_data:
            data = loaded_checkpoints_data[checkpoint_key]
            pop_history = data['pop_history']
            coverage_count_history = count_100_percent_coverage_individuals(pop_history)

            temp_df = pd.DataFrame(coverage_count_history)
            if not temp_df.empty:
                # Rename the count column to the appropriate custom label
                label = labels_to_use.get(checkpoint_key, f"Test {checkpoint_key}")
                temp_df.rename(columns={'100% Coverage Count': label}, inplace=True)
                all_coverage_dfs_for_merge.append(temp_df)
        else:
            print(f"Warning: Checkpoint key {checkpoint_key} not found in loaded_checkpoints_data. Skipping.")

    # Merge all individual DataFrames into a single one
    if all_coverage_dfs_for_merge:
        # Start with the first DataFrame and merge iteratively
        combined_100_coverage_counts_df = all_coverage_dfs_for_merge[0]
        for i in range(1, len(all_coverage_dfs_for_merge)):
            combined_100_coverage_counts_df = pd.merge(
                combined_100_coverage_counts_df,
                all_coverage_dfs_for_merge[i],
                on='generation',
                how='outer' # Use outer merge to include all generations from all checkpoints
            )
        combined_100_coverage_counts_df.sort_values(by='generation', inplace=True)
        combined_100_coverage_counts_df.fillna(0, inplace=True) # Fill NaNs (for generations not present in all) with 0
        combined_100_coverage_counts_df = combined_100_coverage_counts_df.astype(
            {col: int for col in combined_100_coverage_counts_df.columns if col != 'generation'}
        )

        print("\nCombined 100% Coverage Counts per Generation across all checkpoints:")
        display(combined_100_coverage_counts_df)

        # Overwrite the global coverage_count_dataframes variable with the new combined DataFrame
        # This will affect subsequent cells that use coverage_count_dataframes.
        coverage_count_dataframes = combined_100_coverage_counts_df
    else:
        print("No data found to combine for 100% Coverage Counts.")

else:
    print("Error: 'loaded_checkpoints_data' not found or is empty. Please ensure the checkpoints have been loaded.")


Calculating 100% Coverage Counts for each generation across checkpoints and combining into a single DataFrame...

Combined 100% Coverage Counts per Generation across all checkpoints:


,generation,NSGA-NC,NSGA-C50,NSGA-C60,NSGA-C70,NSGA-C80,NSGA-C90,NSGA-C95,NSGA-C99,NSGA-C100,MOGA-WS
0,0,6,6,6,6,6,7,6,6,6,13
1,1,4,8,7,7,19,10,17,22,19,22
2,2,3,5,6,8,16,13,22,33,40,32
3,3,3,5,8,8,11,9,18,26,50,36
4,4,3,4,8,8,8,11,13,26,50,41
...,...,...,...,...,...,...,...,...,...,...,...
96,96,5,7,8,10,8,10,9,12,50,30
97,97,4,7,8,11,8,10,11,11,50,34
98,98,5,9,7,11,8,10,11,11,50,25
99,99,6,8,9,12,8,10,11,11,50,28


In [66]:
export_dataframes_to_csv(combined_100_coverage_counts_df, "/content/vietnam-satellite-simulation/count", "count_pop_100_coverage", False)

✅ Exported count_pop_100_coverage.csv


In [67]:
!git add .
!git commit -am "Đồng bộ format của count_pop_100_coverage"
!git push origin HEAD:moga-simulation-and-extract-plot

[detached HEAD d955fe2] Đồng bộ format của count_pop_100_coverage
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 395 bytes | 395.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/PTNK-ly-tin-2427/vietnam-satellite-simulation.git
   a5f7a20..d955fe2  HEAD -> moga-simulation-and-extract-plot


In [ ]:
# @title
import matplotlib.pyplot as plt
import os

def plot_100_coverage_count_per_checkpoint(coverage_count_dataframes, output_folder):
    """
    Generates plots of the count of individuals with 100% coverage per generation
    for each checkpoint and saves them to the output folder.

    Parameters
    ----------
    coverage_count_dataframes : dict
        A dictionary where keys are checkpoint file paths and values are the
        corresponding DataFrames containing 'Generation' and '100% Coverage Count' columns.
    output_folder : str
        The path to the folder where the plots should be saved.
    """
    if not coverage_count_dataframes:
        print("Không có dữ liệu đếm độ phủ 100% để vẽ biểu đồ.")
        return

    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    sorted_keys = sorted(coverage_count_dataframes.keys())
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(sorted_keys):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(sorted_keys)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # MỚI: Dùng enumerate trên list đã sắp xếp
    for i, checkpoint_key in enumerate(sorted_keys):
        coverage_df = coverage_count_dataframes[checkpoint_key]
        # Check if the required columns exist
        if 'Generation' in coverage_df.columns and '100% Coverage Count' in coverage_df.columns:
            plt.figure(figsize=(10, 6))
            plt.plot(coverage_df['Generation'], coverage_df['100% Coverage Count'], marker='o', linestyle='-')
            plt.gca().spines[['top', 'right']].set_visible(False)

            # MỚI: Quyết định tên cho tiêu đề
            if use_custom_labels:
                checkpoint_name = CUSTOM_LABELS[i]
            else:
                checkpoint_name = f"Test {checkpoint_key}" # Fallback

            plt.xlabel("Generation")
            plt.ylabel("100% Coverage Count")
            # MỚI: Sử dụng checkpoint_name (từ CUSTOM_LABELS) trong tiêu đề
            plt.title(f"100% Coverage Count per Generation ({checkpoint_name})")
            plt.grid(True)

            # Define the filename using the checkpoint key
            filename = f"test_{checkpoint_key}_100_coverage_count_plot.png"
            filepath = os.path.join(output_folder, filename)

            # Save the figure
            plt.savefig(filepath)
            print(f"✅ Saved plot to {filepath}")
            plt.close() # Close the figure to free up memory
        else:
            # MỚI: Cập nhật thông báo lỗi nếu có thể
            label_for_error = CUSTOM_LABELS[i] if use_custom_labels else f"Test {checkpoint_key}"
            print(f"Skipping plot for {label_for_error}: required columns 'Generation' or '100% Coverage Count' not found.")

def plot_100_coverage_count_comparison(coverage_count_dataframes, output_folder):
    """
    Generates a single plot comparing the count of individuals with 100% coverage
    per generation across all loaded checkpoints and saves it to the output folder.

    Parameters
    ----------
    coverage_count_dataframes : dict
        A dictionary where keys are checkpoint file paths and values are the
        corresponding DataFrames containing 'Generation' and '100% Coverage Count' columns.
    output_folder : str
        The path to the folder where the plots should be saved.
    """
    if not coverage_count_dataframes:
        print("Không có dữ liệu đếm độ phủ 100% để vẽ biểu đồ so sánh.")
        return

    # MỚI: Tăng kích thước, thêm markers và linestyles
    plt.figure(figsize=(14, 7))
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']
    plotted_any = False

    # Sắp xếp keys để đảm bảo thứ tự nhất quán
    sorted_keys = sorted(coverage_count_dataframes.keys())

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(sorted_keys):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(sorted_keys)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # MỚI: Dùng enumerate trên list đã sắp xếp
    for i, checkpoint_key in enumerate(sorted_keys):
        coverage_df = coverage_count_dataframes[checkpoint_key]
        # Check if the required columns exist
        if 'Generation' in coverage_df.columns and '100% Coverage Count' in coverage_df.columns:
            # MỚI: Quyết định label
            if use_custom_labels:
                label_name = CUSTOM_LABELS[i]
            else:
                label_name = f"Test {checkpoint_key}" # Fallback

            # MỚI: Chọn marker và linestyle
            marker = markers[i % len(markers)]
            linestyle = linestyles[i % len(linestyles)]

            # MỚI: Cập nhật lệnh plot
            plt.plot(coverage_df['Generation'], coverage_df['100% Coverage Count'],
                     label=label_name,
                     marker=marker,
                     linestyle=linestyle,
                     markersize=5)
            plotted_any = True
        else:
            # MỚI: Cập nhật thông báo lỗi
            label_for_error = CUSTOM_LABELS[i] if use_custom_labels else f"Test {checkpoint_key}"
            print(f"Skipping plot for {label_for_error}: required columns 'Generation' or '100% Coverage Count' not found.")

    if plotted_any:
        plt.gca().spines[['top', 'right']].set_visible(False)
        plt.xlabel("Generation")
        plt.ylabel("100% Coverage Count")
        plt.title("Comparison of 100% Coverage Count per Generation across Checkpoints")

        # MỚI: Cập nhật legend
        plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.grid(True)

        os.makedirs(output_folder, exist_ok=True)

        # Define the filename
        filename = "comparison_100_coverage_count_plot.png"
        filepath = os.path.join(output_folder, filename)

        # Save the figure
        plt.savefig(filepath, bbox_inches='tight') # MỚI: Thêm bbox_inches='tight'
        print(f"✅ Saved comparison plot to {filepath}")
        plt.close() # Close the figure to free up memory
    else:
        print("Không có dữ liệu để vẽ biểu đồ so sánh đếm độ phủ 100%.")


# Example usage (assuming coverage_count_dataframes and OUTPUT_FOLDER are defined in previous cells):
# plot_100_coverage_count_per_checkpoint(coverage_count_dataframes, OUTPUT_FOLDER)
plot_100_coverage_count_comparison(coverage_count_dataframes, OUTPUT_FOLDER)

✅ Saved comparison plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_100_coverage_count_plot.png


In [60]:
# @title
import pandas as pd
import os # Import os module

# Create a dictionary to store the new 100% coverage HOF counts for each checkpoint
new_hof_100_coverage_counts = {}

if 'loaded_checkpoints_data' in locals() and loaded_checkpoints_data:
    print("Calculating new 100% coverage Hall of Fame counts for each checkpoint...")
    # Iterate through the dictionary items (checkpoint_key, data_dict)
    for checkpoint_key, data in loaded_checkpoints_data.items():
        halloffame_history = data['halloffame_history']

        # Create a list to store the counts for the current checkpoint
        checkpoint_counts = []
        previous_hof_100_coverage_set = set() # Keep track of individuals in HOF from previous generation

        for gen_data in halloffame_history:
            generation = gen_data['gen']
            current_halloffame = gen_data['halloffame']
            current_fitnesses = gen_data['fitnesses']

            # Identify individuals with 100% coverage in the current HOF
            current_hof_100_coverage_individuals = [
                tuple(current_halloffame[i]) for i in range(len(current_halloffame))
                if current_fitnesses[i][0] >= 1.0 # Assuming coverage is the first fitness value
            ]

            # Convert to a set for efficient comparison
            current_hof_100_coverage_set = set(current_hof_100_coverage_individuals)

            # Count new individuals with 100% coverage added to HOF in this generation
            # These are individuals in the current set that were NOT in the previous generation's set
            newly_added_100_coverage_count = len(current_hof_100_coverage_set - previous_hof_100_coverage_set)

            # Append the count and generation to the list
            checkpoint_counts.append({'Generation': generation, 'New 100% HOF Count': newly_added_100_coverage_count})

            # Update the set of individuals in HOF for the next generation's comparison
            previous_hof_100_coverage_set = current_hof_100_coverage_set

        # Convert the list of counts to a DataFrame and store it in the dictionary
        # Use the checkpoint_key (integer) as the dictionary key
        new_hof_100_coverage_counts[checkpoint_key] = pd.DataFrame(checkpoint_counts).set_index('Generation')

    # Combine the DataFrames from all checkpoints into a single DataFrame
    # Use concat with axis=1 to join on the 'Generation' index
    if new_hof_100_coverage_counts:
        # Sort the keys before concatenating to ensure consistent column order
        sorted_keys = sorted(new_hof_100_coverage_counts.keys())
        combined_new_hof_100_coverage_df = pd.concat([new_hof_100_coverage_counts[key] for key in sorted_keys], axis=1, keys=sorted_keys)
        # combined_new_hof_100_coverage_df.columns.names = ['Checkpoint Key', 'Metric'] # Add multi-level column names if desired
        print("\nCombined DataFrame of New 100% Coverage Hall of Fame Counts per Generation:")
        display(combined_new_hof_100_coverage_df)
    else:
        print("\nKhông có dữ liệu để tạo DataFrame kết hợp.")

else:
    print("Error: 'loaded_checkpoints_data' not found or is empty. Please ensure the checkpoints have been loaded.")

Calculating new 100% coverage Hall of Fame counts for each checkpoint...

Combined DataFrame of New 100% Coverage Hall of Fame Counts per Generation:


,0,50,60,70,80,90,95,99,100,2016
,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count,New 100% HOF Count
Generation,,,,,,,,,,
0,2,2,2,2,2,2,2,2,2,2
1,2,1,1,1,2,1,1,3,3,1
2,1,3,4,4,5,2,3,3,1,1
3,0,1,3,4,3,2,3,2,6,3
4,0,1,4,0,4,4,0,3,2,1
...,...,...,...,...,...,...,...,...,...,...
96,0,0,0,0,0,0,1,0,0,0
97,0,0,0,2,0,0,1,1,0,0


In [ ]:
# @title
import matplotlib.pyplot as plt
import os

if 'combined_new_hof_100_coverage_df' in locals() and not combined_new_hof_100_coverage_df.empty:
    # MỚI: Tăng kích thước để có không gian cho legend
    plt.figure(figsize=(14, 7))

    # MỚI: Danh sách các hình dáng (markers) và kiểu đường (linestyles)
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']

    # MỚI: Lấy danh sách các cột (checkpoints)
    unique_cols = combined_new_hof_100_coverage_df.columns.get_level_values(0).unique()

    # MỚI: Kiểm tra xem có sử dụng label tùy chỉnh không
    use_custom_labels = False
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(unique_cols):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(unique_cols)}). Sẽ sử dụng tên cột mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên cột mặc định.")

    # MỚI: Plot bằng cách lặp qua enumerate để lấy index (i)
    for i, col_name in enumerate(unique_cols):
        # MỚI: Quyết định label sẽ sử dụng
        if use_custom_labels:
            label_name = CUSTOM_LABELS[i]
        else:
            label_name = col_name # Fallback

        # MỚI: Chọn marker và linestyle
        marker = markers[i % len(markers)]
        linestyle = linestyles[i % len(linestyles)]

        # MỚI: Cập nhật lệnh plot
        plt.plot(combined_new_hof_100_coverage_df.index,
                 combined_new_hof_100_coverage_df[col_name]['New 100% HOF Count'],
                 marker=marker,
                 linestyle=linestyle,
                 label=label_name,
                 markersize=5)

    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.xlabel("Generation")
    plt.ylabel("New 100% Coverage Individuals in Hall of Fame")
    plt.title("New 100% Coverage Individuals Added to Hall of Fame per Generation")

    # MỚI: Cập nhật legend
    plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(True)

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    filename = "comparison_new_hof_100_coverage_plot.png"
    filepath = os.path.join(OUTPUT_FOLDER, filename)

    # MỚI: Thêm bbox_inches='tight'
    plt.savefig(filepath, bbox_inches='tight')
    print(f"✅ Saved plot to {filepath}")
    plt.close() # Close the figure to free up memory
else:
    print("Không có dữ liệu 'combined_new_hof_100_coverage_df' để vẽ biểu đồ.")

✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_new_hof_100_coverage_plot.png


In [68]:
# @title
import pandas as pd
from deap import base, tools, creator # Import necessary DEAP modules

# Check if all_final_hall_of_fame_dfs exists and is not empty
if 'all_final_hall_of_fame_dfs' in locals() and all_final_hall_of_fame_dfs:
    # Concatenate all DataFrames in the list into a single DataFrame
    combined_hall_of_fame_df_raw = pd.concat(all_final_hall_of_fame_dfs, ignore_index=True)

    # Ensure DEAP creator is set up (assuming it was done in a previous cell)
    # If not, you might need to add:
    # creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
    # creator.create("Individual", list, fitness=creator.FitnessMulti)
    # And ensure the evaluate function is registered in the toolbox

    # Create a list of DEAP individuals from the combined DataFrame
    combined_individuals = []
    for index, row in combined_hall_of_fame_df_raw.iterrows():
        # Recreate the individual structure including fitness values
        ind = creator.Individual([
            row['Altitude (km)'],
            row['Inclination (deg)'],
            row['Số mặt phẳng quỹ đạo'],
            row['Số vệ tinh mỗi mặt phẳng'],
            row['Phasing']
        ])
        # Assign the fitness values
        ind.fitness.values = (
            row['Fitness (Coverage)'],
            row['Fitness (Altitude)'],
            row['Fitness (Số vệ tinh)']
        )
        # Assign constraint violation value (assuming it was 0 for HOF individuals, or re-evaluate if needed)
        # For simplicity here, assuming HOF individuals satisfy constraints or we are just comparing fitness
        # If constraints were active and relevant for the final HOF, you'd need to handle cvalues
        # For the purpose of finding the overall Pareto front, we primarily use fitness dominance.
        # If selConstrainedNSGA2 is used, it will implicitly handle the constraints if cvalues are set.
        # Let's assume cvalues were 0 for the HOF individuals from the runs, or re-evaluate if necessary
        # Re-evaluating might be slow, so let's assume cvalues are not strictly needed for this final HOF combination step
        # based on the context that these are already from Pareto fronts.
        # If constraints *must* be considered here, you might need to re-evaluate or store cvalues in the HOF history.
        # Given the previous cells didn't explicitly store cvalues in the HOF history DataFrames,
        # we will proceed using fitness dominance for the combined Pareto front.
        # If the original runs used selConstrainedNSGA2, the HOF should only contain feasible or least-infeasible individuals
        # relative to their own run. Combining them and re-applying constrained selection would refine this.
        # Let's add a placeholder for cvalues if needed, assuming 0 for simplicity based on HOF nature.
        ind.fitness.cvalues = (0,) # Placeholder - adjust if actual constraint violation matters here


        # Add the original checkpoint file information as an attribute to the individual
        # This allows tracking the origin after selection
        ind.checkpoint_file = row['Checkpoint_File']
        ind.original_index = row['Lời giải'] # Keep original index if needed

        combined_individuals.append(ind)

    # Apply the constrained non-dominated sorting (assuming selConstrainedNSGA2 is defined)
    # We want the full Pareto front of the combined individuals
    # k=len(combined_individuals) ensures we get all individuals sorted into fronts
    # The first front is the non-dominated set
    all_fronts = fastConstrainedNondominatedSort(combined_individuals, len(combined_individuals), first_front_only=False)
    combined_pareto_front_individuals = all_fronts[0] # The first front is the desired non-dominated set

    # Convert the resulting Pareto front individuals back to a DataFrame
    combined_hall_of_fame_data = []
    for ind in combined_pareto_front_individuals:
        combined_hall_of_fame_data.append({
            # "case": ind.checkpoint_file, # Get origin from the added attribute
            "case": int(str(ind.checkpoint_file).split('_')[-1].split('.')[0]), # Get origin from the added attribute
            # "Original_Lời giải": ind.original_index, # Get original index
            "coverage": ind.fitness.values[0],
            "altitude": ind[0],
            "num_sats": int(ind.fitness.values[2]),
            "inclinatiom": ind[1],
            "num_planes": int(ind[2]),
            "num_sats_per_plane": int(ind[3]),
            "phasing": ind[4],
            # "Fitness (Altitude)": ind.fitness.values[1],
            # Include cvalues if relevant and stored/reevaluated
            # "Constraint Violation": ind.fitness.cvalues[0] if ind.fitness.cvalues else 0.0,
        })

    combined_hall_of_fame_df = pd.DataFrame(combined_hall_of_fame_data)

    print("✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ tất cả các checkpoint.")
    display(combined_hall_of_fame_df)

else:
    print("⚠️ Không có dữ liệu Hall of Fame cuối cùng nào được tìm thấy để tổng hợp. Vui lòng chạy cell tạo final Hall of Fame DataFrame trước.")

✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ tất cả các checkpoint.


,case,coverage,altitude,num_sats,inclinatiom,num_planes,num_sats_per_plane,phasing
0,0,1.000000,500.0,180,20.0,12,15,2.0
1,0,0.999908,815.0,72,19.0,4,18,1.0
2,0,0.999630,501.0,176,21.0,8,22,1.0
3,50,0.999630,501.0,176,21.0,8,22,1.0
4,60,0.999630,501.0,176,21.0,8,22,1.0
...,...,...,...,...,...,...,...,...
2530,2016,1.000000,533.0,153,20.0,9,17,6.0
2531,2016,0.874249,798.0,44,18.0,11,4,5.0
2532,2016,0.962552,653.0,78,20.0,6,13,3.0
2533,2016,0.909015,509.0,100,20.0,5,20,4.0


In [73]:
# @title
import pandas as pd
from deap import base, tools, creator

# Define the specific checkpoints to combine
selected_checkpoint_keys = [80, 90, 95]

# Filter all_final_hall_of_fame_dfs for the selected checkpoints
selected_final_hof_dfs = {key: all_final_hall_of_fame_dfs[key] for key in selected_checkpoint_keys if key in all_final_hall_of_fame_dfs}

if selected_final_hof_dfs:
    # Concatenate the selected DataFrames into a single raw DataFrame
    combined_top_3_hall_of_fame_df_raw = pd.concat(selected_final_hof_dfs.values(), ignore_index=True)

    # Ensure DEAP creator is set up
    if not hasattr(creator, "FitnessMulti"):
        creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMulti)

    # Create a list of DEAP individuals from the raw combined DataFrame
    combined_top_3_individuals = []
    for index, row in combined_top_3_hall_of_fame_df_raw.iterrows():
        ind = creator.Individual([
            row['Altitude (km)']
            , row['Inclination (deg)']
            , row['Số mặt phẳng quỹ đạo']
            , row['Số vệ tinh mỗi mặt phẳng']
            , row['Phasing']
        ])
        ind.fitness.values = (
            row['Fitness (Coverage)']
            , row['Fitness (Altitude)']
            , row['Fitness (Số vệ tinh)']
        )
        ind.fitness.cvalues = (0,) # Placeholder for constraints

        ind.checkpoint_file = row['Checkpoint_File']
        ind.original_index = row['Lời giải']

        combined_top_3_individuals.append(ind)

    # Apply non-dominated sorting to find the Pareto front of these combined individuals
    all_fronts_top_3 = fastConstrainedNondominatedSort(combined_top_3_individuals, len(combined_top_3_individuals), first_front_only=False)
    combined_top_3_pareto_front_individuals = all_fronts_top_3[0] # The first front is the desired non-dominated set

    # Convert the resulting Pareto front individuals back to a DataFrame
    combined_top_3_hall_of_fame_data = []
    for ind in combined_top_3_pareto_front_individuals:
        combined_top_3_hall_of_fame_data.append({
            # "case": ind.checkpoint_file, # Get origin from the added attribute
            "case": int(str(ind.checkpoint_file).split('_')[-1].split('.')[0]), # Get origin from the added attribute
            # "Original_Lời giải": ind.original_index, # Get original index
            "coverage": ind.fitness.values[0],
            "altitude": ind[0],
            "num_sats": int(ind.fitness.values[2]),
            "inclinatiom": ind[1],
            "num_planes": int(ind[2]),
            "num_sats_per_plane": int(ind[3]),
            "phasing": ind[4],
            # "Fitness (Altitude)": ind.fitness.values[1],
            # Include cvalues if relevant and stored/reevaluated
            # "Constraint Violation": ind.fitness.cvalues[0] if ind.fitness.cvalues else 0.0,
        })

    combined_top_3_hall_of_fame_df = pd.DataFrame(combined_top_3_hall_of_fame_data)

    print(f"✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ các checkpoint {selected_checkpoint_keys}.")
    display(combined_top_3_hall_of_fame_df)
else:
    print(f"⚠️ Không tìm thấy dữ liệu Hall of Fame cho các checkpoint {selected_checkpoint_keys}. Không thể tạo combined_top_3_hall_of_fame_df.")

✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ các checkpoint [80, 90, 95].


,case,coverage,altitude,num_sats,inclinatiom,num_planes,num_sats_per_plane,phasing
0,80,1.000000,598,136,20,8,17,6
1,80,1.000000,622,119,21,7,17,3
2,80,1.000000,714,96,19,4,24,3
3,80,1.000000,722,88,19,4,22,1
4,80,1.000000,772,80,19,4,20,3
...,...,...,...,...,...,...,...,...
1079,95,0.082848,718,4,19,4,1,1
1080,95,0.058992,513,5,17,5,1,2
1081,95,0.052982,667,4,18,4,1,3
1082,95,0.048729,623,4,18,4,1,3


In [ ]:
# @title
import matplotlib.pyplot as plt
import os
import pandas as pd

def plot_all_combined_hof_100_coverage_altitude_vs_satellites_with_duplicates(combined_hall_of_fame_df_raw, output_folder):
    """
    Generates an Altitude vs. Number of Satellites scatter plot for individuals
    with 100% coverage from the raw combined Hall of Fame history (including duplicates).
    (Đã cập nhật để cải thiện visual và dùng CUSTOM_LABELS)
    """
    if combined_hall_of_fame_df_raw is None or combined_hall_of_fame_df_raw.empty:
        print("Không có dữ liệu Hall of Fame tổng hợp thô để vẽ biểu đồ.")
        return

    os.makedirs(output_folder, exist_ok=True)

    coverage_100_percent_individuals = combined_hall_of_fame_df_raw[combined_hall_of_fame_df_raw['Fitness (Coverage)'] >= 1.0].copy()

    if 'Altitude (km)' in coverage_100_percent_individuals.columns and \
       'Fitness (Số vệ tinh)' in coverage_100_percent_individuals.columns and \
       'Checkpoint_File' in coverage_100_percent_individuals.columns and \
       not coverage_100_percent_individuals.empty:

        # MỚI: Tăng kích thước để có không gian cho legend
        plt.figure(figsize=(10, 8))

        checkpoint_files = coverage_100_percent_individuals['Checkpoint_File'].unique()

        colors = plt.cm.get_cmap('tab10', len(checkpoint_files)).colors
        # print(colors)
        markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
        # MỚI: Thêm linestyles
        linestyles = ['-', '--', ':', '-.']

        # MỚI: Logic để sử dụng CUSTOM_LABELS
        # Giả định rằng 'CHECKPOINT_FILES' (list) và 'CUSTOM_LABELS' (list) tồn tại
        # trong global scope và có thứ tự tương ứng.
        use_custom_labels = False
        label_map = {}
        if ('CUSTOM_LABELS' in globals() and 'CHECKPOINT_FILES' in globals() and
            isinstance(CUSTOM_LABELS, (list, tuple)) and isinstance(CHECKPOINT_FILES, (list, tuple)) and
            len(CUSTOM_LABELS) == len(CHECKPOINT_FILES)):

            # Tạo map từ đường dẫn file (key) sang label tùy chỉnh (value)
            label_map = {file_path: label for file_path, label in zip(CHECKPOINT_FILES, CUSTOM_LABELS)}
            use_custom_labels = True
            # print(label_map)
        else:
            print("Cảnh báo: Không tìm thấy 'CUSTOM_LABELS' và 'CHECKPOINT_FILES' (hoặc độ dài không khớp). Sẽ dùng tên file mặc định.")

        for i, cp_file in enumerate(checkpoint_files):
            df_subset = coverage_100_percent_individuals[coverage_100_percent_individuals['Checkpoint_File'] == cp_file].copy()
            df_subset.sort_values(by='Fitness (Số vệ tinh)', inplace=True)

            marker = markers[i % len(markers)]
            # MỚI: Thêm linestyle
            linestyle = linestyles[i % len(linestyles)]
            color = colors[i]

            # MỚI: Quyết định label
            if use_custom_labels:
                # Tìm label từ map, nếu không thấy thì dùng tên file
                label_name = label_map.get(CHECKPOINT_FILES[i], os.path.basename(cp_file))
                # print(label_name)
            else:
                label_name = os.path.basename(cp_file) # Fallback

            # Plot điểm (scatter)
            plt.scatter(df_subset['Fitness (Số vệ tinh)'], df_subset['Altitude (km)'],
                        color=color, marker=marker, label=label_name, alpha=0.7, s=50)

            # Plot đường nối các điểm
            plt.plot(df_subset['Fitness (Số vệ tinh)'], df_subset['Altitude (km)'],
                     color=color, linestyle=linestyle, alpha=0.5) # MỚI: thêm linestyle

        plt.xlabel("Cost")
        plt.ylabel("Altitude (km)")
        plt.title("Trade-off HOF: Độ cao quỹ đạo (km) vs. Số lượng vệ tinh (100% Coverage)")

        # MỚI: Cập nhật legend
        plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.grid(True)

        filename = "all_combined_hof_dup_100_coverage_altitude_vs_satellites_comparison_plot.png"
        filepath = os.path.join(output_folder, filename)

        # MỚI: Cập nhật savefig
        plt.savefig(filepath, bbox_inches='tight')
        print(f"✅ Saved plot to {filepath}")
        plt.close()
    else:
        print("Không có đủ dữ liệu (hoặc cột cần thiết) cho các cá thể có độ phủ 100%. Bỏ qua tạo plot.")


# Example usage
if 'combined_hall_of_fame_df_raw' in locals():
    # Giả định CUSTOM_LABELS và CHECKPOINT_FILES đã được định nghĩa ở các cell trước
    plot_all_combined_hof_100_coverage_altitude_vs_satellites_with_duplicates(combined_hall_of_fame_df_raw, OUTPUT_FOLDER)
else:
    print("⚠️ 'combined_hall_of_fame_df_raw' không được tìm thấy.")

/tmp/ipython-input-3832982151.py:30: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap('tab10', len(checkpoint_files)).colors


✅ Saved plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/all_combined_hof_dup_100_coverage_altitude_vs_satellites_comparison_plot.png


In [92]:
# @title
import pandas as pd
from deap import base, tools, creator
import os
import re

# Ensure DEAP creator is set up
if not hasattr(creator, "FitnessMulti"):
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMulti)

# Master dictionary to store all pairwise combined Hall of Fame DataFrames
# Structure: {base_checkpoint_key: {comparison_name: DataFrame}}
# This will effectively store pairwise_combined_hof_0_dfs, pairwise_combined_hof_10_dfs, etc.
pairwise_combined_hof_dfs = {}

# Check if all_final_hall_of_fame_dfs exists and is not empty
if 'all_final_hall_of_fame_dfs' in locals() and all_final_hall_of_fame_dfs:
    sorted_checkpoint_keys = sorted(all_final_hall_of_fame_dfs.keys())
    if not sorted_checkpoint_keys:
        print("⚠️ Danh sách checkpoint keys rỗng.")
    else:
        # Loop through each checkpoint key to use as the "base" for comparison
        for base_checkpoint_key in sorted_checkpoint_keys:
            hof_base_df = all_final_hall_of_fame_dfs[base_checkpoint_key]

            # Find the original file name for the base checkpoint
            base_filename_full = str(base_checkpoint_key)

            print(f"\n--- Processing pairwise comparisons for base checkpoint: {base_filename_full} ---")

            # Initialize a dictionary for the current base checkpoint's comparisons
            current_base_pairwise_hof_comparisons = {}

            # Loop through all other checkpoint keys for comparison
            for current_checkpoint_key in sorted_checkpoint_keys:
                if base_checkpoint_key == current_checkpoint_key:
                    # Skip comparing a checkpoint with itself
                    continue

                current_hof_df = all_final_hall_of_fame_dfs[current_checkpoint_key]

                # Find the original file name for the current checkpoint
                current_filename_full = str(current_checkpoint_key)

                print(f"  Comparing {base_filename_full} and {current_filename_full}")

                # Combine the dataframes
                combined_df_pairwise_raw = pd.concat([hof_base_df, current_hof_df], ignore_index=True)

                # Create a list of DEAP individuals from the combined DataFrame
                combined_individuals_pairwise = []
                for index, row in combined_df_pairwise_raw.iterrows():
                    ind = creator.Individual([
                        row['Altitude (km)']
                        , row['Inclination (deg)']
                        , row['Số mặt phẳng quỹ đạo']
                        , row['Số vệ tinh mỗi mặt phẳng']
                        , row['Phasing']
                    ])
                    ind.fitness.values = (
                        row['Fitness (Coverage)']
                        , row['Fitness (Altitude)']
                        , row['Fitness (Số vệ tinh)']
                    )
                    ind.fitness.cvalues = (0,) # Placeholder
                    ind.checkpoint_file = row['Checkpoint_File']
                    ind.original_index = row['Lời giải']
                    combined_individuals_pairwise.append(ind)

                # Apply the constrained non-dominated sorting
                all_fronts_pairwise = fastConstrainedNondominatedSort(combined_individuals_pairwise, len(combined_individuals_pairwise), first_front_only=False)
                combined_pareto_front_individuals_pairwise = all_fronts_pairwise[0] # The first front is the desired non-dominated set

                # Convert the resulting Pareto front individuals back to a DataFrame
                combined_hall_of_fame_data_pairwise = []
                for ind in combined_pareto_front_individuals_pairwise:
                    combined_hall_of_fame_data_pairwise.append({
                        # "case": ind.checkpoint_file, # Get origin from the added attribute
                        "case": int(str(ind.checkpoint_file).split('_')[-1].split('.')[0]), # Get origin from the added attribute
                        # "Original_Lời giải": ind.original_index, # Get original index
                        "coverage": ind.fitness.values[0],
                        "altitude": ind[0],
                        "num_sats": int(ind.fitness.values[2]),
                        "inclinatiom": ind[1],
                        "num_planes": int(ind[2]),
                        "num_sats_per_plane": int(ind[3]),
                        "phasing": ind[4],
                        # "Fitness (Altitude)": ind.fitness.values[1],
                        # Include cvalues if relevant and stored/reevaluated
                        # "Constraint Violation": ind.fitness.cvalues[0] if ind.fitness.cvalues else 0.0,
                    })

                combined_hall_of_fame_df_pairwise = pd.DataFrame(combined_hall_of_fame_data_pairwise)

                # Store the pairwise combined DataFrame in the current base checkpoint's comparisons dictionary
                comparison_key = f"{base_filename_full}_vs_{current_filename_full}"
                current_base_pairwise_hof_comparisons[comparison_key] = combined_hall_of_fame_df_pairwise

            # Assign the comparisons for the current base checkpoint to the master dictionary
            pairwise_combined_hof_dfs[base_checkpoint_key] = current_base_pairwise_hof_comparisons

else:
    print("⚠️ Không có dữ liệu Hall of Fame cuối cùng nào được tìm thấy để tổng hợp. Vui lòng chạy cell tạo final Hall of Fame DataFrame trước.")

# Print summary of the created structure
print("\n--- Summary of pairwise_combined_hof_dfs ---")
for base_key, comparisons in pairwise_combined_hof_dfs.items():
    print(f"Base Checkpoint Key: {base_key}")
    for comp_name, df in comparisons.items():
        print(f"  - Comparison: {comp_name}, Num individuals: {len(df)}")


--- Processing pairwise comparisons for base checkpoint: 0 ---
  Comparing 0 and 50
  Comparing 0 and 60
  Comparing 0 and 70
  Comparing 0 and 80
  Comparing 0 and 90
  Comparing 0 and 95
  Comparing 0 and 99
  Comparing 0 and 100
  Comparing 0 and 2016

--- Processing pairwise comparisons for base checkpoint: 50 ---
  Comparing 50 and 0
  Comparing 50 and 60
  Comparing 50 and 70
  Comparing 50 and 80
  Comparing 50 and 90
  Comparing 50 and 95
  Comparing 50 and 99
  Comparing 50 and 100
  Comparing 50 and 2016

--- Processing pairwise comparisons for base checkpoint: 60 ---
  Comparing 60 and 0
  Comparing 60 and 50
  Comparing 60 and 70
  Comparing 60 and 80
  Comparing 60 and 90
  Comparing 60 and 95
  Comparing 60 and 99
  Comparing 60 and 100
  Comparing 60 and 2016

--- Processing pairwise comparisons for base checkpoint: 70 ---
  Comparing 70 and 0
  Comparing 70 and 50
  Comparing 70 and 60
  Comparing 70 and 80
  Comparing 70 and 90
  Comparing 70 and 95
  Comparing 70 and

In [102]:
# @title
import pandas as pd
import numpy as np
import re

# 1. Initialize an empty list to store the rows of the new DataFrame
combined_analysis_data = []

# 2. Create a mapping from checkpoint keys (integers) to their respective CUSTOM_LABELS.
checkpoint_label_map = {}
# Ensure CHECKPOINT_FILES and CUSTOM_LABELS are available from previous cells
if 'CHECKPOINT_FILES' in globals() and 'CUSTOM_LABELS' in globals():
    if len(CHECKPOINT_FILES) == len(CUSTOM_LABELS):
        for i, key in enumerate(CHECKPOINT_FILES):
            checkpoint_label_map[key] = CUSTOM_LABELS[i]
    else:
        print("Warning: Length of CHECKPOINT_FILES and CUSTOM_LABELS do not match. Using raw keys.")
        for key in CHECKPOINT_FILES:
            checkpoint_label_map[key] = f"Test {key}"
else:
    print("Warning: CHECKPOINT_FILES or CUSTOM_LABELS not found. Using raw keys.")
    # Fallback if global variables are not set
    # This part depends on the structure of `analysis_dfs` keys to extract available keys dynamically
    all_keys = set()
    for comp_key in analysis_dfs.keys():
        match = re.match(r'test_(\d+)_vs_test_(\d+)', comp_key)
        if match:
            all_keys.add(int(match.group(1)))
            all_keys.add(int(match.group(2)))
    for key in sorted(list(all_keys)):
        checkpoint_label_map[key] = f"Test {key}"

# Create a list of all CUSTOM_LABELS for DataFrame columns, sorted by their original key order
sorted_custom_labels = [checkpoint_label_map[key] for key in sorted(checkpoint_label_map.keys())]

# 3. Iterate through the analysis_dfs dictionary
if 'analysis_dfs' in locals() and analysis_dfs:
    for comparison_key, comparison_df in analysis_dfs.items():
        # a. Extract the two checkpoint keys from the comparison_key string
        match = re.match(r'test_(\d+)_vs_test_(\d+)', comparison_key)
        if not match:
            print(f"Skipping invalid comparison_key format: {comparison_key}")
            continue

        key1 = int(match.group(1))
        key2 = int(match.group(2))

        # b. Get the corresponding CUSTOM_LABELS for key1 and key2
        label1 = checkpoint_label_map.get(key1, f'Test {key1}')
        label2 = checkpoint_label_map.get(key2, f'Test {key2}')

        # Column names in comparison_df use 'Test X Proportion'
        df_col1_name = f"Test {key1} Proportion"
        df_col2_name = f"Test {key2} Proportion"

        # c. Extract the proportions
        # Full Pareto Front Proportion
        full_prop1 = comparison_df.loc[comparison_df.index.str.contains("Full Pareto Front Proportion"), df_col1_name].iloc[0] if df_col1_name in comparison_df.columns else np.nan
        full_prop2 = comparison_df.loc[comparison_df.index.str.contains("Full Pareto Front Proportion"), df_col2_name].iloc[0] if df_col2_name in comparison_df.columns else np.nan

        # 100% Coverage Proportion
        cov100_prop1 = comparison_df.loc[comparison_df.index.str.contains("100% Coverage Proportion"), df_col1_name].iloc[0] if df_col1_name in comparison_df.columns else np.nan
        cov100_prop2 = comparison_df.loc[comparison_df.index.str.contains("100% Coverage Proportion"), df_col2_name].iloc[0] if df_col2_name in comparison_df.columns else np.nan

        # d. For the 'Full Pareto Front Proportion':
        row_full = {lbl: np.nan for lbl in sorted_custom_labels}
        row_full[label1] = full_prop1
        row_full[label2] = full_prop2
        row_full['Comparison'] = f'{label1} vs {label2} (Full Pareto Front)'
        combined_analysis_data.append(row_full)

        # e. Repeat for '100% Coverage Proportion'
        row_100_coverage = {lbl: np.nan for lbl in sorted_custom_labels}
        row_100_coverage[label1] = cov100_prop1
        row_100_coverage[label2] = cov100_prop2
        row_100_coverage['Comparison'] = f'{label1} vs {label2} (100% Coverage)'
        combined_analysis_data.append(row_100_coverage)

# 4. Create a pandas DataFrame from combined_analysis_data
combined_pairwise_analysis_df = pd.DataFrame(combined_analysis_data).set_index('Comparison')

print("Combined pairwise analysis DataFrame created successfully.")
display(combined_pairwise_analysis_df)

Combined pairwise analysis DataFrame created successfully.


,NSGA-NC,NSGA-C50,NSGA-C60,NSGA-C70,NSGA-C80,NSGA-C90,NSGA-C95,NSGA-C99,NSGA-C100,MOGA-WS
Comparison,,,,,,,,,,
NSGA-NC vs NSGA-C50 (Full Pareto Front),0.570597,0.429403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NSGA-NC vs NSGA-C50 (100% Coverage),0.400000,0.600000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NSGA-NC vs NSGA-C60 (Full Pareto Front),0.587290,NaN,0.412710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NSGA-NC vs NSGA-C60 (100% Coverage),0.388889,NaN,0.611111,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NSGA-NC vs NSGA-C70 (Full Pareto Front),0.602790,NaN,NaN,0.39721,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
MOGA-WS vs NSGA-C95 (100% Coverage),NaN,NaN,NaN,NaN,NaN,NaN,0.8,NaN,NaN,0.200000
MOGA-WS vs NSGA-C99 (Full Pareto Front),NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.862745,NaN,0.137255
MOGA-WS vs NSGA-C99 (100% Coverage),NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.833333,NaN,0.166667


In [106]:
# @title
import pandas as pd
import numpy as np
from deap import base, tools, creator

# Define the specific checkpoints to combine
selected_checkpoint_keys_three = [80, 90, 95]

# Filter all_final_hall_of_fame_dfs for the selected checkpoints
selected_final_hof_dfs_three = {key: all_final_hall_of_fame_dfs[key] for key in selected_checkpoint_keys_three if key in all_final_hall_of_fame_dfs}

if selected_final_hof_dfs_three:
    # Concatenate the selected DataFrames into a single raw DataFrame
    combined_three_cases_hall_of_fame_df_raw = pd.concat(selected_final_hof_dfs_three.values(), ignore_index=True)

    # Ensure DEAP creator is set up
    if not hasattr(creator, "FitnessMulti"):
        creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0, -1.0), cvalues=tuple)
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMulti)

    # Create a list of DEAP individuals from the raw combined DataFrame
    combined_three_cases_individuals = []
    for index, row in combined_three_cases_hall_of_fame_df_raw.iterrows():
        ind = creator.Individual([
            row['Altitude (km)']
            , row['Inclination (deg)']
            , row['Số mặt phẳng quỹ đạo']
            , row['Số vệ tinh mỗi mặt phẳng']
            , row['Phasing']
        ])
        ind.fitness.values = (
            row['Fitness (Coverage)']
            , row['Fitness (Altitude)']
            , row['Fitness (Số vệ tinh)']
        )
        ind.fitness.cvalues = (0,) # Placeholder for constraints

        # Add the original checkpoint file information as an attribute to the individual
        ind.checkpoint_key = row['Checkpoint_Key'] # Store the integer key directly
        combined_three_cases_individuals.append(ind)

    # Apply non-dominated sorting to find the Pareto front of these combined individuals
    all_fronts_three_cases = fastConstrainedNondominatedSort(combined_three_cases_individuals, len(combined_three_cases_individuals), first_front_only=False)
    combined_three_cases_pareto_front_individuals = all_fronts_three_cases[0] # The first front is the desired non-dominated set

    # Convert the resulting Pareto front individuals back to a DataFrame for analysis
    combined_three_cases_pareto_front_df = pd.DataFrame([
        {
            "checkpoint_key": ind.checkpoint_key,
            "coverage": ind.fitness.values[0],
            "altitude": ind.fitness.values[1],
            "num_sats": ind.fitness.values[2]
        }
        for ind in combined_three_cases_pareto_front_individuals
    ])

    print(f"✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ các checkpoint {selected_checkpoint_keys_three}.")
    # display(combined_three_cases_pareto_front_df.head())

    # --- Calculate Proportions ---
    total_pf_individuals = len(combined_three_cases_pareto_front_df)
    total_100_coverage_pf_individuals = len(combined_three_cases_pareto_front_df[combined_three_cases_pareto_front_df['coverage'] >= 1.0])

    # Prepare data for the final DataFrame
    analysis_results = {
        'Metric': [
            f'Full Combined Pareto Front Proportion ({total_pf_individuals} individuals)',
            f'100% Coverage Combined Pareto Front Proportion ({total_100_coverage_pf_individuals} individuals)'
        ]
    }

    for key in selected_checkpoint_keys_three:
        label = CUSTOM_LABELS[CHECKPOINT_FILES.index(key)] if key in CHECKPOINT_FILES else f"Test {key}"

        # Count individuals from current key in the full combined PF
        count_in_full_pf = len(combined_three_cases_pareto_front_df[combined_three_cases_pareto_front_df['checkpoint_key'] == key])
        prop_in_full_pf = count_in_full_pf / total_pf_individuals if total_pf_individuals > 0 else 0.0

        # Count individuals from current key in the 100% coverage combined PF
        count_in_100_cov_pf = len(combined_three_cases_pareto_front_df[
            (combined_three_cases_pareto_front_df['checkpoint_key'] == key) &
            (combined_three_cases_pareto_front_df['coverage'] >= 1.0)
        ])
        prop_in_100_cov_pf = count_in_100_cov_pf / total_100_coverage_pf_individuals if total_100_coverage_pf_individuals > 0 else 0.0

        analysis_results[label] = [prop_in_full_pf, prop_in_100_cov_pf]

    combined_three_cases_proportions_df = pd.DataFrame(analysis_results).set_index('Metric')

    print("\nProportions of each case in the combined Pareto Front:")
    display(combined_three_cases_proportions_df)

else:
    print(f"⚠️ Không tìm thấy dữ liệu Hall of Fame cho các checkpoint {selected_checkpoint_keys_three}. Không thể tạo DataFrame so sánh.")


✅ Đã tạo Hall of Fame tổng hợp (Pareto Front) từ các checkpoint [80, 90, 95].

Proportions of each case in the combined Pareto Front:


,NSGA-C80,NSGA-C90,NSGA-C95
Metric,,,
Full Combined Pareto Front Proportion (1084 individuals),0.442804,0.288745,0.268450
100% Coverage Combined Pareto Front Proportion (21 individuals),0.476190,0.238095,0.285714


In [74]:
# @title
import pandas as pd
import os # Import os module

# This will store individual DataFrames for each checkpoint (similar to simpsons_diversity_dataframes)
hof_100_coverage_counts_individual_dfs = {}
# This will be the single combined DataFrame as requested by the user
combined_hof_100_coverage_df = None
# This list will hold temporary DataFrames to be merged into the combined one
all_hof_100_coverage_dfs_for_merge = []

if 'loaded_checkpoints_data' in locals() and loaded_checkpoints_data:
    print("Calculating 100% coverage Hall of Fame counts for each checkpoint...")

    # Ensure CUSTOM_LABELS and CHECKPOINT_FILES are available and correctly mapped
    if 'CUSTOM_LABELS' not in globals() or 'CHECKPOINT_FILES' not in globals() or \
       len(CUSTOM_LABELS) != len(CHECKPOINT_FILES):
        print("Warning: CUSTOM_LABELS or CHECKPOINT_FILES not properly defined or lengths mismatch. Using default keys for columns.")
        # Fallback to using checkpoint_key as column name if CUSTOM_LABELS/CHECKPOINT_FILES are not aligned
        sorted_checkpoint_keys = sorted(loaded_checkpoints_data.keys())
        labels_to_use = {key: f"Test {key}" for key in sorted_checkpoint_keys}
    else:
        # Create a mapping from checkpoint key (integer) to its CUSTOM_LABEL
        labels_to_use = {key: label for key, label in zip(CHECKPOINT_FILES, CUSTOM_LABELS)}
        sorted_checkpoint_keys = CHECKPOINT_FILES # Use the provided order from CHECKPOINT_FILES

    for checkpoint_key in sorted_checkpoint_keys:
        if checkpoint_key in loaded_checkpoints_data:
            data = loaded_checkpoints_data[checkpoint_key]
            halloffame_history = data['halloffame_history']

            # Create a list to store the counts for the current checkpoint
            checkpoint_counts = []

            for gen_data in halloffame_history:
                generation = gen_data['gen']
                current_halloffame = gen_data['halloffame']
                current_fitnesses = gen_data['fitnesses']

                # Count individuals with 100% coverage in the current HOF
                count_100_coverage = sum(1 for i in range(len(current_halloffame))
                                           if current_fitnesses[i][0] >= 1.0) # Assuming coverage is the first fitness value

                # Append the count and generation to the list
                checkpoint_counts.append({'generation': generation, '100% HOF Coverage Count': count_100_coverage})

            temp_df = pd.DataFrame(checkpoint_counts)
            if not temp_df.empty:
                # Store individual DF in the dictionary
                hof_100_coverage_counts_individual_dfs[checkpoint_key] = temp_df.copy()

                # Prepare for the combined DataFrame: rename '100% HOF Coverage Count' to its custom label
                label = labels_to_use.get(checkpoint_key, f"Test {checkpoint_key}")
                temp_df_for_merge = temp_df.rename(columns={'100% HOF Coverage Count': label})
                all_hof_100_coverage_dfs_for_merge.append(temp_df_for_merge)
        else:
            print(f"Warning: Checkpoint key {checkpoint_key} not found in loaded_checkpoints_data. Skipping.")

    # --- Create the single combined DataFrame as requested by the user ---
    if all_hof_100_coverage_dfs_for_merge:
        # Start with the first DataFrame and merge iteratively
        combined_hof_100_coverage_df = all_hof_100_coverage_dfs_for_merge[0]
        for i in range(1, len(all_hof_100_coverage_dfs_for_merge)):
            combined_hof_100_coverage_df = pd.merge(
                combined_hof_100_coverage_df,
                all_hof_100_coverage_dfs_for_merge[i],
                on='generation',
                how='outer' # Use outer merge to include all generations from all checkpoints
            )
        combined_hof_100_coverage_df.sort_values(by='generation', inplace=True)
        combined_hof_100_coverage_df.fillna(0, inplace=True) # Fill NaNs (for generations not present in all) with 0
        # Convert count columns to integer type after filling NaNs
        combined_hof_100_coverage_df = combined_hof_100_coverage_df.astype(
            {col: int for col in combined_hof_100_coverage_df.columns if col != 'generation'}
        )

        print("\nCombined 100% HOF Coverage Counts per Generation across all checkpoints:")
        display(combined_hof_100_coverage_df)

        # Overwrite the global hof_100_coverage_counts variable with the new combined DataFrame
        # This will affect subsequent cells that use hof_100_coverage_counts if they expect a single DF.
        hof_100_coverage_counts = combined_hof_100_coverage_df
    else:
        print("No data found to combine for 100% HOF Coverage Counts.")

else:
    print("Error: 'loaded_checkpoints_data' not found or is empty. Please ensure the checkpoints have been loaded.")

Calculating 100% coverage Hall of Fame counts for each checkpoint...

Combined 100% HOF Coverage Counts per Generation across all checkpoints:


,generation,NSGA-NC,NSGA-C50,NSGA-C60,NSGA-C70,NSGA-C80,NSGA-C90,NSGA-C95,NSGA-C99,NSGA-C100,MOGA-WS
0,0,2,2,2,2,2,2,2,2,2,2
1,1,4,3,3,3,3,3,3,4,4,3
2,2,3,4,6,6,7,3,5,5,4,3
3,3,3,5,7,9,9,4,6,6,9,6
4,4,3,4,10,9,8,8,6,8,11,7
...,...,...,...,...,...,...,...,...,...,...,...
96,96,11,24,12,24,16,17,18,15,7,11
97,97,11,24,12,22,16,17,19,11,7,11
98,98,11,24,12,23,16,14,19,12,7,11
99,99,11,24,14,23,16,14,17,12,7,11


In [ ]:
# @title
import matplotlib.pyplot as plt
import os

def plot_hof_100_coverage_count_comparison(hof_100_coverage_counts, output_folder):
    """
    Generates a single line plot comparing the count of individuals with 100% coverage
    in the Hall of Fame per generation across all loaded checkpoints.
    (Đã cập nhật để cải thiện visual và dùng CUSTOM_LABELS)
    """
    if not hof_100_coverage_counts:
        print("Không có dữ liệu đếm độ phủ 100% trong Hall of Fame để vẽ biểu đồ so sánh.")
        return

    # MỚI: Tăng kích thước, thêm markers và linestyles
    plt.figure(figsize=(14, 7))
    markers = ['o', 's', '^', 'v', 'D', 'p', '*', '+', 'x']
    linestyles = ['-', '--', ':', '-.']
    plotted_any = False

    # Sắp xếp keys để đảm bảo thứ tự nhất quán
    sorted_keys = sorted(hof_100_coverage_counts.keys())

    # MỚI: Kiểm tra CUSTOM_LABELS
    use_custom_labels = False
    if 'CUSTOM_LABELS' in globals() and isinstance(CUSTOM_LABELS, (list, tuple)):
        if len(CUSTOM_LABELS) == len(sorted_keys):
            use_custom_labels = True
        else:
            print(f"Cảnh báo: Số lượng CUSTOM_LABELS ({len(CUSTOM_LABELS)}) không khớp với số lượng checkpoint ({len(sorted_keys)}). Sẽ sử dụng tên mặc định.")
    else:
        print("Cảnh báo: Biến CUSTOM_LABELS không tìm thấy. Sẽ sử dụng tên mặc định.")

    # MỚI: Dùng enumerate trên list đã sắp xếp
    for i, checkpoint_key in enumerate(sorted_keys):
        hof_coverage_df = hof_100_coverage_counts[checkpoint_key]

        if 'Generation' in hof_coverage_df.columns and '100% HOF Coverage Count' in hof_coverage_df.columns:
            # MỚI: Quyết định label
            if use_custom_labels:
                label_name = CUSTOM_LABELS[i]
            else:
                label_name = f"Test {checkpoint_key}" # Fallback

            # MỚI: Chọn marker và linestyle
            marker = markers[i % len(markers)]
            linestyle = linestyles[i % len(linestyles)]

            # MỚI: Cập nhật lệnh plot
            plt.plot(hof_coverage_df['Generation'], hof_coverage_df['100% HOF Coverage Count'],
                     label=label_name,
                     marker=marker,
                     linestyle=linestyle,
                     markersize=5)
            plotted_any = True
        else:
            label_for_error = CUSTOM_LABELS[i] if use_custom_labels else f"Test {checkpoint_key}"
            print(f"Skipping plot for {label_for_error}: required columns 'Generation' or '100% HOF Coverage Count' not found.")

    if plotted_any:
        plt.gca().spines[['top', 'right']].set_visible(False)
        plt.xlabel("Generation")
        plt.ylabel("100% HOF Coverage Count")
        plt.title("Comparison of 100% Coverage Individuals in Hall of Fame per Generation")

        # MỚI: Cập nhật legend
        plt.legend(title="Case", bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.grid(True)

        os.makedirs(output_folder, exist_ok=True)

        filename = "comparison_hof_100_coverage_count_plot.png"
        # MỚI: Sửa lỗi (sử dụng 'output_folder' thay vì 'OUTPUT_FOLDER' toàn cục)
        filepath = os.path.join(output_folder, filename)

        # MỚI: Cập nhật savefig
        plt.savefig(filepath, bbox_inches='tight')
        print(f"✅ Saved comparison plot to {filepath}")
        plt.close()
    else:
        print("Không có dữ liệu để vẽ biểu đồ so sánh đếm độ phủ 100% trong Hall of Fame.")


# Example usage (assuming hof_100_coverage_counts, OUTPUT_FOLDER,
# and CUSTOM_LABELS are defined in previous cells):
#
plot_hof_100_coverage_count_comparison(hof_100_coverage_counts, OUTPUT_FOLDER)

✅ Saved comparison plot to /content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 15/comparison_hof_100_coverage_count_plot.png


## **10. Xuất Kết quả TLE để Mô phỏng**

In [84]:
import os

# Define source and destination directories
source_dir = "/content/vietnam-satellite-simulation/count_100_coverage"
dest_dir = os.path.join(source_dir, "data")

# Create the destination directory if it doesn't exist
!mkdir -p "{dest_dir}"

# Move all files and folders from source_dir to dest_dir,
# excluding the dest_dir itself to prevent errors.
# find command ensures we only move direct children and exclude 'data'
!find "{source_dir}" -maxdepth 1 -mindepth 1 -not -name "data" -exec mv -t "{dest_dir}" {} +

print(f"✅ All files and folders moved from '{source_dir}' to '{dest_dir}'.")

find: ‘{source_dir}’: No such file or directory
✅ All files and folders moved from '/content/vietnam-satellite-simulation/count_100_coverage' to '/content/vietnam-satellite-simulation/count_100_coverage/data'.


### **10.1. Hướng dẫn Xuất TLE**
Bạn có thể chọn bất kỳ lời giải nào từ các DataFrame đã tạo ở trên (`hall_of_fame_df` hoặc `history_hall_of_fame_df`) để xuất TLE.
1.  **Chọn một hàng (lời giải):** Lấy ra hàng tương ứng với cấu hình bạn muốn.
2.  **Trích xuất các tham số:** Lấy các giá trị từ các cột `Altitude`, `Inclination`, `Số mặt phẳng quỹ đạo`, v.v.
3.  **Gọi hàm `generate_tle_string()`:** Truyền các tham số đã trích xuất vào hàm này để tạo ra chuỗi TLE hoàn chỉnh.
4.  **Lưu vào tệp:** Ghi chuỗi TLE ra một tệp văn bản (ví dụ: `constellation.tle`).

In [ ]:
# @title
def generate_tle_string(individual, filename="constellation.tle"):
  """
  Tạo tệp TLE từ cấu hình chòm sao.

  params = [altitude_km, inclination_deg, num_planes, sats_per_plane, phasing]

  Returns:
      str: Tên tệp TLE đã tạo.
  """
  altitude_km, inclination_deg, num_planes, sats_per_plane, phasing = individual
  epoch_year, epoch_day = tle_epoch_from_datetime(start_time)
  mean_motion = mean_motion_from_altitude(altitude_km)
  eccentricity = 0.0
  arg_perigee_deg = 0.0
  satnum_base = 10000
  tle_lines = []
  for p in range(int(num_planes)):
        RAAN = p * 360 / num_planes
        for s in range(int(sats_per_plane)):
            mean_anomaly = (360 / sats_per_plane) * (
                s + phasing * p / num_planes
            ) % 360
            satnum = satnum_base + p * int(sats_per_plane) + s + 1
            line1, line2 = generate_tle_celestrak(
                satnum=satnum,
                epoch_year=epoch_year,
                epoch_day=epoch_day,
                inclination_deg=inclination_deg,
                raan_deg=RAAN,
                eccentricity=eccentricity,
                arg_perigee_deg=arg_perigee_deg,
                mean_anomaly_deg=mean_anomaly,
                mean_motion_rev_per_day=mean_motion,
                rev_number=1
            )
            name = f"SAT_{satnum}"
            tle_lines.append(name + "\n")
            tle_lines.append(line1 + "\n")
            tle_lines.append(line2 + "\n")

  # Ensure output folder exists
  os.makedirs(OUTPUT_FOLDER, exist_ok=True)
  full_filepath = os.path.join(OUTPUT_FOLDER, filename)
  with open(full_filepath, "w") as f:
      f.writelines(tle_lines)

  return full_filepath

In [ ]:
generate_tle_string([665, 19, 5, 21, 2], 'tle_95_665')
generate_tle_string([500, 20, 12, 15, 2], 'tle_0_500')

'/content/drive/MyDrive/PTNK/NCKH/Sprint 11/output/test 13/tle_0_500'

In [ ]:
# @title
import pandas as pd
import os

def export_dataframes_to_csv(data_structure, folder, prefix="", is_dict=True):
    """
    Exports pandas DataFrames to CSV files.

    Parameters
    ----------
    data_structure : dict or pandas.DataFrame
        A dictionary of DataFrames or a single DataFrame to export.
    folder : str
        The output folder path.
    prefix : str
        A prefix to prepend to the filenames.
    is_dict : bool
        True if data_structure is a dictionary of DataFrames, False if it's a single DataFrame.
    """
    os.makedirs(folder, exist_ok=True)

    if is_dict:
        for key, df in data_structure.items():
            if isinstance(df, pd.DataFrame) and not df.empty:
                filename = os.path.join(folder, f"{prefix}_{key}.csv")
                df.to_csv(filename, index=True)
                print(f"✅ Exported {prefix}_{key}.csv")
            else:
                print(f"⚠️ Skipping {prefix}_{key}: Not a DataFrame or is empty.")
    else:
        if isinstance(data_structure, pd.DataFrame) and not data_structure.empty:
            filename = os.path.join(folder, f"{prefix}.csv")
            data_structure.to_csv(filename, index=True)
            print(f"✅ Exported {prefix}.csv")
        else:
            print(f"⚠️ Skipping {prefix}: Not a DataFrame or is empty.")


print("--- Exporting DataFrames ---")

# Export logbook_dataframes
if 'logbook_dataframes' in locals() and logbook_dataframes:
    export_dataframes_to_csv(logbook_dataframes, OUTPUT_FOLDER, "logbook_stats")

# Export all_history_hall_of_fame_dfs
if 'all_history_hall_of_fame_dfs' in locals() and all_history_hall_of_fame_dfs:
    export_dataframes_to_csv(all_history_hall_of_fame_dfs, OUTPUT_FOLDER, "hof_history")

# Export all_hall_of_fame_gene_stats_dfs
if 'all_hall_of_fame_gene_stats_dfs' in locals() and all_hall_of_fame_gene_stats_dfs:
    export_dataframes_to_csv(all_hall_of_fame_gene_stats_dfs, OUTPUT_FOLDER, "hof_gene_stats")

# Export all_final_hall_of_fame_dfs
if 'all_final_hall_of_fame_dfs' in locals() and all_final_hall_of_fame_dfs:
    export_dataframes_to_csv(all_final_hall_of_fame_dfs, OUTPUT_FOLDER, "final_hof")

# Export simpsons_diversity_dataframes
if 'simpsons_diversity_dataframes' in locals() and simpsons_diversity_dataframes:
    export_dataframes_to_csv(simpsons_diversity_dataframes, OUTPUT_FOLDER, "simpsons_diversity")

# Export coverage_count_dataframes
if 'coverage_count_dataframes' in locals() and coverage_count_dataframes:
    export_dataframes_to_csv(coverage_count_dataframes, OUTPUT_FOLDER, "coverage_count")

# Export hof_100_coverage_counts
if 'hof_100_coverage_counts' in locals() and hof_100_coverage_counts:
    export_dataframes_to_csv(hof_100_coverage_counts, OUTPUT_FOLDER, "hof_100_coverage_count")

# Export new_hof_100_coverage_counts (individual checkpoint data, if needed)
# If combined_new_hof_100_coverage_df covers this, can skip individual ones.
# For now, exporting both for completeness.
if 'new_hof_100_coverage_counts' in locals() and new_hof_100_coverage_counts:
    export_dataframes_to_csv(new_hof_100_coverage_counts, OUTPUT_FOLDER, "new_hof_100_coverage_per_checkpoint")

# Export combined_new_hof_100_coverage_df (single DF)
if 'combined_new_hof_100_coverage_df' in locals() and combined_new_hof_100_coverage_df is not None:
    export_dataframes_to_csv(combined_new_hof_100_coverage_df, OUTPUT_FOLDER, "combined_new_hof_100_coverage", is_dict=False)

# Export combined_hall_of_fame_df (single DF)
if 'combined_hall_of_fame_df' in locals() and combined_hall_of_fame_df is not None:
    export_dataframes_to_csv(combined_hall_of_fame_df, OUTPUT_FOLDER, "overall_combined_pareto_front", is_dict=False)

# Export combined_top_3_hall_of_fame_df (single DF)
if 'combined_top_3_hall_of_fame_df' in locals() and combined_top_3_hall_of_fame_df is not None:
    export_dataframes_to_csv(combined_top_3_hall_of_fame_df, OUTPUT_FOLDER, "combined_top_3_pareto_front", is_dict=False)

# Export overall_proportions_df (single DF)
if 'overall_proportions_df' in locals() and overall_proportions_df is not None:
    export_dataframes_to_csv(overall_proportions_df, OUTPUT_FOLDER, "overall_pareto_proportions", is_dict=False)

# Export analysis_dfs (dict of DFs - pairwise comparisons)
if 'analysis_dfs' in locals() and analysis_dfs:
    export_dataframes_to_csv(analysis_dfs, OUTPUT_FOLDER, "pairwise_analysis")

print("--- DataFrame export complete ---")

--- Exporting DataFrames ---
✅ Exported logbook_stats_0.csv
✅ Exported logbook_stats_10.csv
✅ Exported logbook_stats_20.csv
✅ Exported logbook_stats_30.csv
✅ Exported logbook_stats_40.csv
✅ Exported logbook_stats_50.csv
✅ Exported logbook_stats_60.csv
✅ Exported logbook_stats_70.csv
✅ Exported logbook_stats_80.csv
✅ Exported logbook_stats_90.csv
✅ Exported logbook_stats_95.csv
✅ Exported logbook_stats_99.csv
✅ Exported logbook_stats_100.csv
✅ Exported logbook_stats_2016.csv
✅ Exported hof_history_0.csv
✅ Exported hof_history_10.csv
✅ Exported hof_history_20.csv
✅ Exported hof_history_30.csv
✅ Exported hof_history_40.csv
✅ Exported hof_history_50.csv
✅ Exported hof_history_60.csv
✅ Exported hof_history_70.csv
✅ Exported hof_history_80.csv
✅ Exported hof_history_90.csv
✅ Exported hof_history_95.csv
✅ Exported hof_history_99.csv
✅ Exported hof_history_100.csv
✅ Exported hof_history_2016.csv
✅ Exported hof_gene_stats_0.csv
✅ Exported hof_gene_stats_10.csv
✅ Exported hof_gene_stats_20.csv
✅

In [86]:
# @title

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure CUSTOM_LABELS is available for plotting
if 'CUSTOM_LABELS' not in globals():
    CUSTOM_LABELS = [
        "NSGA-II (No Constraint)",
        "NSGA-II (f₁ ≥ 10%)",
        "NSGA-II (f₁ ≥ 20%)",
        "NSGA-II (f₁ ≥ 30%)",
        "NSGA-II (f₁ ≥ 40%)",
        "NSGA-II (f₁ ≥ 50%)",
        "NSGA-II (f₁ ≥ 60%)",
        "NSGA-II (f₁ ≥ 70%)",
        "NSGA-II (f₁ ≥ 80%)",
        "NSGA-II (f₁ ≥ 90%)",
        "NSGA-II (f₁ ≥ 95%)",
        "NSGA-II (f₁ ≥ 99%)",
        "NSGA-II (f₁ = 100%)",
        "MOGA-WS (Meziane-Tani 2016)"
    ]

# Ensure CHECKPOINT_FILES is available
if 'CHECKPOINT_FILES' not in globals():
    CHECKPOINT_FILES = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100, 2016]

if 'analysis_dfs' in locals() and analysis_dfs:
    num_checkpoints = len(CHECKPOINT_FILES)
    checkpoint_keys = sorted(CHECKPOINT_FILES) # Sort keys for consistent matrix order

    # Initialize matrices for win rates
    win_rate_matrix_full = np.zeros((num_checkpoints, num_checkpoints))
    win_rate_matrix_100_coverage = np.zeros((num_checkpoints, num_checkpoints))

    # Create labels for the heatmap
    heatmap_labels = [f"Test {key}" for key in checkpoint_keys]
    if len(CUSTOM_LABELS) == len(checkpoint_keys):
        heatmap_labels = CUSTOM_LABELS

    print("Building win rate matrices...")

    for i, key1 in enumerate(checkpoint_keys):
        for j, key2 in enumerate(checkpoint_keys):
            if i == j:
                # A checkpoint against itself will have a win rate of 0.5 (or 1.0 if we consider it always "wins" itself, but 0.5 is more neutral)
                win_rate_matrix_full[i, j] = 0.5
                win_rate_matrix_100_coverage[i, j] = 0.5
                continue

            comparison_key_forward = f"test_{key1}_vs_test_{key2}"
            comparison_key_backward = f"test_{key2}_vs_test_{key1}" # Also consider the reverse for consistency

            # Try to get data for key1 vs key2
            if comparison_key_forward in analysis_dfs:
                df_analysis = analysis_dfs[comparison_key_forward]
                # Extract proportions for key1
                col1_name = f"Test {key1} Proportion"
                if col1_name in df_analysis.columns:
                    win_rate_matrix_full[i, j] = df_analysis.loc[df_analysis.index.str.contains("Full Pareto Front Proportion"), col1_name].iloc[0]
                    win_rate_matrix_100_coverage[i, j] = df_analysis.loc[df_analysis.index.str.contains("100% Coverage Proportion"), col1_name].iloc[0]
                else:
                    print(f"Warning: Column '{col1_name}' not found in {comparison_key_forward}. Setting win rate to 0.")
                    win_rate_matrix_full[i, j] = 0.0
                    win_rate_matrix_100_coverage[i, j] = 0.0
            else:
                 # If the forward key is not found, it implies that the 'analysis_dfs' might have been created from one direction only
                 # Or the entry was missing for some reason. For now, set to 0.
                 # In a full setup, we'd ideally ensure analysis_dfs contains both directions.
                print(f"Warning: Comparison '{comparison_key_forward}' not found in analysis_dfs. Setting win rate to 0.")
                win_rate_matrix_full[i, j] = 0.0
                win_rate_matrix_100_coverage[i, j] = 0.0

    print("Win rate matrices built.")

    # Convert to DataFrames for better heatmap plotting
    win_rate_df_full = pd.DataFrame(win_rate_matrix_full, index=heatmap_labels, columns=heatmap_labels)
    win_rate_df_100_coverage = pd.DataFrame(win_rate_matrix_100_coverage, index=heatmap_labels, columns=heatmap_labels)

    # --- Plotting Heatmaps ---
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    def plot_heatmap(df, title, filename):
        plt.figure(figsize=(12, 10))
        sns.heatmap(df, annot=True, cmap='viridis', fmt=".2f", linewidths=.5, cbar_kws={'label': 'Proportion of Pareto Front Individuals'})
        plt.title(title)
        plt.xlabel("So với Test (Đối thủ)")
        plt.ylabel("Test của bạn (Người chiến thắng)")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        filepath = os.path.join(OUTPUT_FOLDER, filename)
        plt.savefig(filepath)
        print(f"✅ Saved heatmap to {filepath}")
        plt.close()

    # Plot for Full Pareto Front
    plot_heatmap(win_rate_df_full, "Tỷ lệ thắng trong tập Pareto đầy đủ (full Pareto Front)", "win_rate_heatmap_full_pareto.png")

    # Plot for 100% Coverage Pareto Front
    plot_heatmap(win_rate_df_100_coverage, "Tỷ lệ thắng trong tập Pareto với độ phủ 100% (100% Coverage Pareto Front)", "win_rate_heatmap_100_coverage_pareto.png")

else:
    print("⚠️ 'analysis_dfs' not found or is empty. Cannot create win rate matrices. Please ensure previous steps ran correctly.")

⚠️ 'analysis_dfs' not found or is empty. Cannot create win rate matrices. Please ensure previous steps ran correctly.


In [88]:
!git add .
!git commit -am "Tạo folder data để chứa tất cả csv và file google collab (chưa hoàn thiện) tạo ra chúng nó"
!git push origin HEAD:moga-simulation-and-extract-plot --force

HEAD detached from a106948
nothing to commit, working tree clean
Enumerating objects: 219, done.
Counting objects: 100% (219/219), done.
Delta compression using up to 2 threads
Compressing objects: 100% (179/179), done.
Writing objects: 100% (218/218), 4.75 MiB | 2.19 MiB/s, done.
Total 218 (delta 53), reused 150 (delta 36), pack-reused 0
remote: Resolving deltas: 100% (53/53), completed with 1 local object.
To https://github.com/PTNK-ly-tin-2427/vietnam-satellite-simulation.git
 + b803492...9158e3d HEAD -> moga-simulation-and-extract-plot (forced update)
